# PIMCO AI Surveillance Analysis Based on ABS Prospectuses

The saved cell outputs are historical. Run the notebook in order to see results from the current code.

### Milestones
- **Milestone 1:** Data collection, parsing, metadata filtering, and keyword/regex search
- **Milestone 2:** Embedding-based semantic search and Q&A
- **Milestone 3:** User interface and evaluation


The primary interface is the standalone Streamlit application. The Gradio cells are a deprecated fallback.

In [75]:
import os
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup as bsoup

In [76]:
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "manifest.csv").is_file()
     and (path / "notebooks").is_dir()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start the notebook from the project directory.")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import prospectus_evaluation as evaluation
from prospectus_provenance import validate_manifest

print("Project root:", PROJECT_ROOT)

Project root: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses


In [77]:
raw_data = PROJECT_ROOT / "data" / "raw"
processed_data = PROJECT_ROOT / "data" / "processed"
manifest_path = PROJECT_ROOT / "data" / "manifest.csv"

processed_data.mkdir(parents=True, exist_ok=True)

# Output file
output = processed_data / "abs_index.csv"

print("Raw data:", raw_data)
print("Manifest:", manifest_path)
print("Processed data:", processed_data)
print("Output file:", output)

Raw data: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/raw
Manifest: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/manifest.csv
Processed data: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed
Output file: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_index.csv


## Milestone 1 — Data Collection and Text Extraction

The dataset contains SEC ABS prospectus filings in HTML (`.htm`) format.

For each filing, we:
1. Locate the HTML file
2. Read the HTML
3. Remove HTML formatting
4. Extract readable text
5. Store the result in a DataFrame

In [78]:
def get_htm_files(data_folder):
    data_folder = Path(data_folder)
    
    if not data_folder.exists():
        raise FileNotFoundError(
            f"Data folder not found: {data_folder}"
        )
    
    htm_files = sorted(
        file
        for file in data_folder.iterdir()
        if file.is_file() and file.suffix.lower() in {".htm", ".html"}
    )
    
    return htm_files


htm_files = get_htm_files(raw_data)

print(f"Found {len(htm_files)} HTM filings.")

Found 122 HTM filings.


In [79]:
# Extract readable text from an SEC HTML filing.    
def extract_text_from_htm(file_path):
   with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:
        
        html = file.read()
        soup = bsoup(
        html,
        "html.parser"
    )
        text = soup.get_text(
        " ",
        strip=True
    )
        return text

In [80]:
data = []

for file_path in htm_files:
    text = extract_text_from_htm(
        file_path
    )
    
    data.append({
        "file_name": file_path.name,
        "extracted_text": text
    })

df = pd.DataFrame(data)

print("Number of filings:", len(df))
print("Number of columns:", len(df.columns))

df.head()

Number of filings: 122
Number of columns: 2


,file_name,extracted_text
0,AFS_SENSUB_CORP__2024-05-15_424H.htm,424H 1 d679938d424h.htm 424H 424H Table of Con...
1,AFS_SENSUB_CORP__2024-06-26_424H.htm,424H 1 d792078d424h.htm 424H 424H Table of Con...
2,AFS_SENSUB_CORP__2024-10-02_424H.htm,424H 1 d847362d424h.htm 424H 424H Table of Con...
3,AFS_SENSUB_CORP__2025-01-06_424H.htm,424H 1 d909372d424h.htm 424H 424H Table of Con...
4,AFS_SENSUB_CORP__2025-05-01_424H.htm,424H 1 d111258d424h.htm 424H 424H Table of Con...


In [81]:
# metadata
def filename_metadata(df):
    df = df.copy()
    # Filing date
    df["filing_date"] = df["file_name"].str.extract(
        r"(\d{4}-\d{2}-\d{2})"
    )
    # Filing type
    df["filing_type"] = df["file_name"].str.extract(
        r"_(424H)\.html?$"
    )
    # Deal name
    df["deal_name"] = (
        df["file_name"]
        .str.replace(
            r"_\d{4}-\d{2}-\d{2}_424H\.html?$",
            "",
            regex=True
        )
        .str.replace(
            "_",
            " ",
            regex=False
        )
    )
    return df
df = filename_metadata(df)
df[
    [
        "file_name",
        "deal_name",
        "filing_date",
        "filing_type"
    ]
].tail()

,file_name,deal_name,filing_date,filing_type
117,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2025-10-01_424...,WORLD OMNI AUTO RECEIVABLES LLC,2025-10-01,424H
118,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-02-02_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-02-02,424H
119,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-04-29_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-04-29,424H
120,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-07-13_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-07-13,424H
121,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-08-12_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-08-12,424H


In [82]:
# Read manifest csv
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Manifest not found: {manifest_path}"
    )
manifest = pd.read_csv(
    manifest_path
)
print("Manifest rows:", len(manifest))
print("Manifest columns:")
print(manifest.columns.tolist())

# Get actual filename from local_path
manifest["file_name"] = (
    manifest["local_path"]
    .astype(str)
    .str.split(r"[\\/]")
    .str[-1]
)

Manifest rows: 129
Manifest columns:
['entity_name', 'cik', 'form', 'filing_date', 'accession', 'accession_clean', 'primary_doc', 'description', 'doc_url', 'index_url', 'local_path', 'downloaded']


In [83]:
manifest["file_name"] = (
    manifest["local_path"]
    .astype(str)
    .str.split(r"[\\/]")
    .str[-1]
)

In [84]:
# Remove old SEC metadata columns from df
metadata_columns = [
    "cik",
    "accession",
    "primary_doc",
    "doc_url",
    "cik_x",
    "cik_y",
    "accession_x",
    "accession_y",
    "primary_doc_x",
    "primary_doc_y",
    "doc_url_x",
    "doc_url_y"
]
df = df.drop(
    columns=[
        col for col in metadata_columns
        if col in df.columns
    ],
    errors="ignore"
)

# Resolve collisions from the retained HTML. Unresolved rows remain unmatched.
manifest_unique, provenance_collisions = validate_manifest(manifest, raw_data)
if provenance_collisions:
    display(pd.DataFrame(provenance_collisions))

# Merge SEC metadata into df
df = df.merge(
    manifest_unique[
        [
            "file_name",
            "cik",
            "accession",
            "primary_doc",
            "doc_url"
        ]
    ],
    on="file_name",
    how="left"
    )
print(
        "SEC metadata matched:",
        df["cik"].notna().sum(),
        "/",
        len(df)
    )

Unique manifest filings: 122
SEC metadata matched: 122 / 122


In [85]:
# Normalize whitespace in extracted filing text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    return text.strip()

In [86]:
df["clean_text"] = (
    df["extracted_text"]
    .apply(clean_text)
)

In [87]:
# Character count
df["text_length"] = (
    df["clean_text"]
    .str.len()
)
# Word count
df["word_count"] = (
    df["clean_text"]
    .str.split()
    .str.len()
)
# Convert filing date to datetime
df["filing_date"] = pd.to_datetime(
    df["filing_date"],
    errors="coerce"
)
# Extract filing year
df["filing_year"] = (
    df["filing_date"]
    .dt.year
)

In [88]:

print("DATA QUALITY CHECK: ")
print(
    "Total filings:",
    len(df)
)
print(
    "Empty extracted text:",
    (df["text_length"] == 0).sum()
)
print(
    "Missing filing dates:",
    df["filing_date"].isna().sum()
)
print(
    "Missing deal names:",
    df["deal_name"].isna().sum()
)
print(
    "Missing CIK:",
    df["cik"].isna().sum()
)
print(
    "Missing SEC URLs:",
    df["doc_url"].isna().sum()
)

print(
    "Average words per filing:",
    round(df["word_count"].mean())
)
print(
    "Shortest filing:",
    df["word_count"].min(),
    "words"
)
print(
    "Longest filing:",
    df["word_count"].max(),
    "words"
)

DATA QUALITY CHECK: 
Total filings: 122
Empty extracted text: 0
Missing filing dates: 0
Missing deal names: 0
Missing CIK: 0
Missing SEC URLs: 0
Average words per filing: 147154
Shortest filing: 83055 words
Longest filing: 586179 words


In [89]:
if not output.exists():
    df.to_csv(output, index=False)
print(
    f"Processed index:\n{output}"
)
print(
    f"Number of filings: {len(df)}"
)
print(
    f"Number of columns: {len(df.columns)}"
)

Saved processed index to:
/Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_index.csv
Number of filings: 122
Number of columns: 13


In [90]:
def extract_snippets(
    text,
    pattern,
    snippets_per_result=2,
    context_chars=250
):
    """
    Extract text surrounding each search match.
    """
    
    if not isinstance(text, str):
        return []
    
    try:
        compiled_pattern = re.compile(
            pattern,
            flags=re.IGNORECASE
        )
    except re.error:
        return []
    
    snippets = []
    
    for match in compiled_pattern.finditer(text):
        
        start = max(
            0,
            match.start() - context_chars
        )
        
        end = min(
            len(text),
            match.end() + context_chars
        )
        
        snippet = text[start:end].strip()
        
        if start > 0:
            snippet = "..." + snippet
        
        if end < len(text):
            snippet += "..."
        
        snippets.append(snippet)
        
        if len(snippets) >= snippets_per_result:
            break
    
    return snippets

In [91]:
def search_filings(
    query,
    year=None,
    filing_type=None,
    regex=False,
    max_results=10,
    snippets_per_result=2
):
    """
    Search ABS prospectuses using keywords,
    phrases, or regular expressions.
    """
    
    # Start with the processed dataset
    search_df = df.copy()
    
    # Metadata filtering
    if year is not None:
        
        search_df = search_df[
            search_df["filing_year"] == int(year)
        ]
    
    if filing_type is not None:
        
        search_df = search_df[
            search_df["filing_type"]
            .fillna("")
            .str.lower()
            ==
            str(filing_type).lower()
        ]
    
    # Create search pattern
    if regex:
        pattern = query
    
    else:
        pattern = re.escape(query)
    
    try:
        compiled_pattern = re.compile(
            pattern,
            flags=re.IGNORECASE
        )
    
    except re.error as error:
        raise ValueError(
            f"Invalid regular expression: {error}"
        )

    # Count matches
    search_df["matches"] = (
        search_df["clean_text"]
        .fillna("")
        .apply(
            lambda text:
            len(
                compiled_pattern.findall(text)
            )
        )
    )
    
    # Keep only matching filings
    search_df = search_df[
        search_df["matches"] > 0
    ].copy()
    
    # Supporting snippets
    search_df["supporting_snippets"] = (
        search_df["clean_text"]
        .fillna("")
        .apply(
            lambda text:
            extract_snippets(
                text,
                pattern,
                snippets_per_result
            )
        )
    )
    
    # Rank results
    search_df = search_df.sort_values(
        by="matches",
        ascending=False
    )
    
    # Return useful columns
    columns = [
        "deal_name",
        "filing_date",
        "filing_year",
        "filing_type",
        "matches",
        "supporting_snippets",
        "doc_url",
        "file_name"
    ]
    
    return (
        search_df[columns]
        .head(max_results)
        .reset_index(drop=True)
    )

In [92]:
results = search_filings(
    "custody",
    max_results=10
)
results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-03-23,2026,424H,15,[...assu Whole Loans 240 Certain Rights of eac...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-15,2026,424H,13,[...ssu Whole Loans 246 Certain Rights of each...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
6,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-05,2026,424H,10,[...Passu Whole Loans 243 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
7,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
8,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-02-11,2026,424H,9,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
9,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-06-10,2026,424H,8,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...


In [93]:
results = search_filings(
    "physical custody",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name


In [94]:
results = search_filings(
    r"\b(physical|paper)\b.{0,100}\b(custody|custodian)\b",
    regex=True,
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,SANTANDER DRIVE AUTO RECEIVABLES LLC,2026-02-11,2026,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2026-02-1...
1,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-11-06,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-11-0...
2,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-09-10,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-09-1...
3,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-07-17,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-07-1...
4,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-05-14,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-05-1...
5,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-03-12,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-03-1...
6,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-01-10,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-01-1...
7,Bridgecrest Auto Funding LLC,2026-07-15,2026,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/197482...,Bridgecrest_Auto_Funding_LLC_2026-07-15_424H.htm
8,SANTANDER DRIVE AUTO RECEIVABLES LLC,2024-10-10,2024,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2024-10-1...
9,SANTANDER DRIVE AUTO RECEIVABLES LLC,2024-09-11,2024,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2024-09-1...


In [95]:
results = search_filings(
    "custody",
    year=2025,
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,BMW AUTO LEASING LLC,2025-05-29,2025,424H,6,[...s 128 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-05-29_424H.htm
6,BMW AUTO LEASING LLC,2025-10-02,2025,424H,6,[...s 129 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-10-02_424H.htm
7,HARLEY-DAVIDSON CUSTOMER FUNDING CORP,2025-05-14,2025,424H,3,[...e Uniform Commercial Code. As described un...,https://www.sec.gov/Archives/edgar/data/111492...,HARLEY-DAVIDSON_CUSTOMER_FUNDING_CORP__2025-05...
8,CARMAX AUTO FUNDING LLC,2025-09-09,2025,424H,2,[...r of this securitization transaction and w...,https://www.sec.gov/Archives/edgar/data/125938...,CARMAX_AUTO_FUNDING_LLC_2025-09-09_424H.htm
9,TOYOTA AUTO FINANCE RECEIVABLES LLC,2025-10-08,2025,424H,2,[...behalf of their participants through their...,https://www.sec.gov/Archives/edgar/data/113113...,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2025-10-08...


In [96]:
results = search_filings(
    "custody",
    filing_type="424H",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-03-23,2026,424H,15,[...assu Whole Loans 240 Certain Rights of eac...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-15,2026,424H,13,[...ssu Whole Loans 246 Certain Rights of each...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
6,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-05,2026,424H,10,[...Passu Whole Loans 243 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
7,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
8,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-02-11,2026,424H,9,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
9,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-06-10,2026,424H,8,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...


In [97]:
results = search_filings(
    "custody",
    year=2025,
    filing_type="424H",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,BMW AUTO LEASING LLC,2025-05-29,2025,424H,6,[...s 128 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-05-29_424H.htm
6,BMW AUTO LEASING LLC,2025-10-02,2025,424H,6,[...s 129 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-10-02_424H.htm
7,HARLEY-DAVIDSON CUSTOMER FUNDING CORP,2025-05-14,2025,424H,3,[...e Uniform Commercial Code. As described un...,https://www.sec.gov/Archives/edgar/data/111492...,HARLEY-DAVIDSON_CUSTOMER_FUNDING_CORP__2025-05...
8,CARMAX AUTO FUNDING LLC,2025-09-09,2025,424H,2,[...r of this securitization transaction and w...,https://www.sec.gov/Archives/edgar/data/125938...,CARMAX_AUTO_FUNDING_LLC_2025-09-09_424H.htm
9,TOYOTA AUTO FINANCE RECEIVABLES LLC,2025-10-08,2025,424H,2,[...behalf of their participants through their...,https://www.sec.gov/Archives/edgar/data/113113...,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2025-10-08...


In [98]:
test_queries = [
    "physical",
    "paper",
    "custody",
    "custodian"
]

for query in test_queries:
    
    results = search_filings(
        query,
        max_results=5
    )
    
    print(f"QUERY: {query}")
    print(f"MATCHING FILINGS: {len(results)}")
    
    if len(results) > 0:
        
        top_result = results.iloc[0]
        
        print(
            "TOP DEAL:",
            top_result["deal_name"]
        )
        
        print(
            "DATE:",
            top_result["filing_date"]
        )
        
        print(
            "MATCHES:",
            top_result["matches"]
        )
        
        print("\nSNIPPET:")
        
        print(
            top_result["supporting_snippets"][0]
        )
        
    else:
        
        print("No results found.")

QUERY: physical
MATCHING FILINGS: 5
TOP DEAL: WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC
DATE: 2026-02-11 00:00:00
MATCHES: 59

SNIPPET:
...ptible to damage or wear and tear than other types of multifamily housing, the reliance on the financial well-being of the college or university to which it relates, competition from on campus housing units, which may adversely affect occupancy, the physical layout of the housing, which may not be readily convertible to traditional multifamily use, and that student tenants have a higher turnover rate than other types of multifamily tenants, which in certain cases is compounded by the fact that student l...
QUERY: paper
MATCHING FILINGS: 5
TOP DEAL: BMW AUTO LEASING LLC
DATE: 2025-05-29 00:00:00
MATCHES: 19

SNIPPET:
...eness of the settlement pending PIMCO’s appeal of its objection. The case has been referred to the circuit mediator, and, on February 24, 2025, the Consumer Financial Protection Bureau sought extension of time for 90 days to prep

In [99]:
df.head()
df.columns.tolist()

['file_name',
 'extracted_text',
 'filing_date',
 'filing_type',
 'deal_name',
 'cik',
 'accession',
 'primary_doc',
 'doc_url',
 'clean_text',
 'text_length',
 'word_count',
 'filing_year']

## Milestone 2

In [100]:
import pandas as pd
import re
import json
import gzip
import hashlib
from pathlib import Path

# Load Milestone 1 data
csv_index_path = processed_data / "abs_index.csv"

df = pd.read_csv(csv_index_path)

print("Total filings:", len(df))
print("Columns:", df.columns.tolist())

Total filings: 122
Columns: ['file_name', 'extracted_text', 'filing_date', 'filing_type', 'deal_name', 'cik', 'accession', 'primary_doc', 'doc_url', 'clean_text', 'text_length', 'word_count', 'filing_year']


In [101]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75

def chunk_document(row, chunk_size=500, overlap=75):
    text = str(row["clean_text"])

    if not text.strip() or text == "nan":
        return []

    # Find each word and its position in the original text
    words = list(re.finditer(r"\S+", text))
    total_words = len(words)

    step = chunk_size - overlap

    # Create a unique ID for each filing
    filename = str(row["file_name"])
    accession = str(row["accession"])

    file_hash = hashlib.sha256(
        filename.encode("utf-8")
    ).hexdigest()[:10]

    filing_id = f"{accession}_{file_hash}"

    chunks = []
    previous_end = 0

    for start in range(0, total_words, step):
        end = min(start + chunk_size, total_words)

        # Avoid creating a very small final chunk
        if (
            end == total_words
            and total_words > chunk_size
            and end - start < chunk_size // 2
        ):
            start = total_words - chunk_size

        if end <= previous_end:
            break

        # Get character positions
        char_start = words[start].start()
        char_end = words[end - 1].end()

        chunk_text = text[char_start:char_end]

        chunk = {
            "chunk_id": f"{filing_id}_chunk_{len(chunks)}",
            "filing_id": filing_id,
            "chunk_index": len(chunks),
            "file_name": filename,
            "deal_name": row["deal_name"],
            "filing_date": str(row["filing_date"]),
            "filing_year": int(row["filing_year"]),
            "filing_type": row["filing_type"],
            "cik": str(row["cik"]),
            "accession": accession,
            "doc_url": row["doc_url"],
            "word_start": start,
            "word_end": end,
            "char_start": char_start,
            "char_end": char_end,
            "word_count": end - start,
            "text": chunk_text
        }

        chunks.append(chunk)
        previous_end = end

        if end == total_words:
            break

    return chunks

In [102]:
chunk_archive_path = processed_data / "abs_chunks.jsonl.gz"

# Reuse existing chunks to preserve the FAISS checkpoint
REBUILD_CHUNKS = False

if chunk_archive_path.exists() and not REBUILD_CHUNKS:
    print("Reusing existing chunk file:", chunk_archive_path)
    print("No chunks regenerated.")

else:
    faiss_path = processed_data / "abs_openai_semantic.faiss"
    state_file = processed_data / "abs_openai_semantic_state.json"
    
    if faiss_path.exists() or state_file.exists():
        raise RuntimeError(
            "A FAISS checkpoint already exists. "
            "Do not regenerate chunks without verifying "
            "or rebuilding their embeddings."
        )

    total_chunks = 0
    filings_processed = 0
    empty_filings = 0

    with gzip.open(chunk_archive_path, "wt", encoding="utf-8") as f:
        for _, row in df.iterrows():
            document_chunks = chunk_document(
                row,
                chunk_size=CHUNK_SIZE,
                overlap=CHUNK_OVERLAP
            )

            if not document_chunks:
                empty_filings += 1
                continue

            assert document_chunks[0]["word_start"] == 0
            assert document_chunks[-1]["word_end"] == len(
                str(row["clean_text"]).split()
            )

            for chunk in document_chunks:
                f.write(
                    json.dumps(
                        chunk,
                        ensure_ascii=False,
                        default=str
                    ) + "\n"
                )

            total_chunks += len(document_chunks)
            filings_processed += 1

    print("Filings processed:", filings_processed)
    print("Total chunks generated:", total_chunks)
    print("Empty filings:", empty_filings)
    print("Saved chunks to:", chunk_archive_path)

Reusing existing chunk file: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_chunks.jsonl.gz
No chunks regenerated.


In [103]:
with gzip.open(
    chunk_archive_path, "rt", encoding="utf-8"
) as f:

    for i in range(3):
        chunk = json.loads(next(f))

        print("Chunk ID:", chunk["chunk_id"])
        print("Deal:", chunk["deal_name"])
        print("Filing date:", chunk["filing_date"])
        print("Word count:", chunk["word_count"])
        print("Source:", chunk["doc_url"])
        print()
        print("Text preview:")
        print(chunk["text"][:500])
        print()

Chunk ID: 0001193125-24-139394_6460085a42_chunk_0
Deal: AFS SENSUB CORP 
Filing date: 2024-05-15
Word count: 500
Source: https://www.sec.gov/Archives/edgar/data/1347185/000119312524139394/d679938d424h.htm

Text preview:
424H 1 d679938d424h.htm 424H 424H Table of Contents Filed pursuant to Rule 424(h) Registration Statement No. 333-261851 Information contained herein is subject to completion or amendment. A registration statement relating to these securities has been filed and declared effective with the Securities and Exchange Commission. These securities may not be sold without delivery of a final prospectus. This prospectus shall not constitute an offer to sell or the solicitation of an offer to buy nor shall

Chunk ID: 0001193125-24-139394_6460085a42_chunk_1
Deal: AFS SENSUB CORP 
Filing date: 2024-05-15
Word count: 500
Source: https://www.sec.gov/Archives/edgar/data/1347185/000119312524139394/d679938d424h.htm

Text preview:
- ● excess cashflow collected on the pool of automobile lo

In [104]:
chunk_ids = set()
total = 0
empty_chunks = 0
missing_urls = 0
duplicate_ids = 0

with gzip.open(
    chunk_archive_path, "rt", encoding="utf-8"
) as f:

    for line in f:
        chunk = json.loads(line)
        total += 1

        if not chunk["text"].strip():
            empty_chunks += 1

        if not chunk["doc_url"]:
            missing_urls += 1

        if chunk["chunk_id"] in chunk_ids:
            duplicate_ids += 1

        chunk_ids.add(chunk["chunk_id"])

print("Total chunks:", total)
print("Empty chunks:", empty_chunks)
print("Missing source URLs:", missing_urls)
print("Duplicate chunk IDs:", duplicate_ids)

assert empty_chunks == 0
assert missing_urls == 0
assert duplicate_ids == 0


Total chunks: 42283
Empty chunks: 0
Missing source URLs: 0
Duplicate chunk IDs: 0


In [105]:
%pip install -r "{PROJECT_ROOT / 'requirements.txt'}"


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [106]:
import os
import json
import gzip
import hashlib

from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import tiktoken

from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

# Existing Milestone 1 directory
processed_data = Path(processed_data).resolve()

# Repository root
repo_root = processed_data.parents[1]

# Load PIMCO API key from .env
load_dotenv(repo_root / ".env")

api_key = os.getenv("API_KEY") or os.getenv("OPENAI_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("OPENAI_BASE_URL") or None,
    max_retries=0,
    timeout=90.0
) if api_key else None

def require_api_client():
    if client is None:
        raise RuntimeError("Configure API_KEY or OPENAI_API_KEY for new API queries.")
    return client

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

print("API client configured:", client is not None)
print("Embedding model:", EMBEDDING_MODEL)
print("Processed data directory:", processed_data)


API credentials loaded successfully.
Embedding model: text-embedding-3-small
Processed data directory: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed


In [107]:
chunk_path = processed_data / "abs_chunks.jsonl.gz"

chunks = []

with gzip.open(chunk_path, "rt", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

# Toyota April 2026 filing
TOYOTA_FILE = (
    "TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_"
    "2026-04-08_424H.htm"
)

toyota_chunks = [
    c for c in chunks
    if c["file_name"] == TOYOTA_FILE
]

other_chunks = [
    c for c in chunks
    if c["file_name"] != TOYOTA_FILE
]

# IMPORTANT: Keep this ordering unchanged.
# It matches the previously saved FAISS index.
ordered_chunks = toyota_chunks + other_chunks

print("Total chunks:", len(ordered_chunks))
print("Toyota chunks:", len(toyota_chunks))
print("Other chunks:", len(other_chunks))
print("First filing:", ordered_chunks[0]["file_name"])

Total chunks: 42283
Toyota chunks: 312
Other chunks: 41971
First filing: TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08_424H.htm


In [108]:
import json
import hashlib
import faiss

index_path = processed_data / "abs_openai_semantic.faiss"
state_path = processed_data / "abs_openai_semantic_state.json"

source_hash = hashlib.sha256(chunk_path.read_bytes()).hexdigest()

order_hash = hashlib.sha256(
    "\n".join(
        c["chunk_id"] for c in ordered_chunks
    ).encode("utf-8")
).hexdigest()

if index_path.exists() and state_path.exists():
    state = json.loads(state_path.read_text())
    index = faiss.read_index(str(index_path))

    # Check the existing embedding checkpoint
    assert state["model"] == EMBEDDING_MODEL
    assert state["order_hash"] == order_hash, ("Chunk order changed. Do not reuse the index.")
    assert index.ntotal == state["indexed_chunks"] <= len(ordered_chunks)
    assert index.d == EMBEDDING_DIM

    if state["source_hash"] != source_hash:
        raise RuntimeError(
            "Checkpoint source hash differs from the chunk archive. "
            "Verify the matching archive before reusing this index."
        )

    print("Existing FAISS index loaded successfully.")

elif not index_path.exists() and not state_path.exists():
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    print("Created new FAISS index.")

else:
    raise RuntimeError("Incomplete checkpoint. Check saved index files.")

print("\nIndexed chunks:", index.ntotal)
print("Total available chunks:", len(ordered_chunks))
print("Remaining:", len(ordered_chunks) - index.ntotal)
print("Embedding dimensions:", index.d)

print(
    "Toyota chunks indexed:",
    min(index.ntotal, len(toyota_chunks)),
    "/",
    len(toyota_chunks)
)

Existing FAISS index loaded successfully.

Indexed chunks: 2808
Total available chunks: 42283
Remaining: 39475
Embedding dimensions: 1536
Toyota chunks indexed: 312 / 312


In [109]:
# Keep False unless you want to generate more embeddings
RUN_MORE_EMBEDDINGS = False

# None means all remaining chunks
TARGET_CHUNKS = None

BATCH_SIZE = 24
SAVE_EVERY = 240
MAX_RUN_TOKENS = 1_000_000

def save_checkpoint():
    faiss.write_index(index, str(index_path))

    state = {
        "model": EMBEDDING_MODEL,
        "source_hash": source_hash,
        "order_hash": order_hash,
        "indexed_chunks": index.ntotal
    }

    state_path.write_text(
        json.dumps(state, indent=2)
    )

if RUN_MORE_EMBEDDINGS:
    encoding = tiktoken.encoding_for_model(EMBEDDING_MODEL)
    target = (
        len(ordered_chunks)
        if TARGET_CHUNKS is None
        else min(TARGET_CHUNKS, len(ordered_chunks))
    )

    start_count = index.ntotal
    last_saved = index.ntotal

    estimated_run_tokens = 0
    actual_run_tokens = 0

    for start in tqdm(
        range(index.ntotal, target, BATCH_SIZE),
        desc="Creating OpenAI embeddings"
    ):

        end = min(start + BATCH_SIZE, target)

        batch_texts = [
            ordered_chunks[i]["text"]
            for i in range(start, end)
        ]

        batch_token_counts = [
            len(encoding.encode(
                text,
                disallowed_special=()
            ))
            for text in batch_texts
        ]

        assert max(batch_token_counts) <= 8192, (
            "Chunk exceeds embedding token limit."
        )

        batch_tokens = sum(batch_token_counts)

        if estimated_run_tokens + batch_tokens > MAX_RUN_TOKENS:
            print("Paused: per-run token budget reached.")
            break

        # API request
        response = require_api_client().embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch_texts
        )

        response_data = sorted(
            response.data,
            key=lambda item: item.index
        )

        vectors = np.array(
            [item.embedding for item in response_data],
            dtype="float32"
        )

        assert vectors.shape == (
            len(batch_texts), EMBEDDING_DIM
        )

        # Normalize for cosine similarity
        faiss.normalize_L2(vectors)

        # Add new embeddings to existing index
        index.add(vectors)

        estimated_run_tokens += batch_tokens

        if response.usage is not None:
            actual_run_tokens += response.usage.total_tokens

        # Save periodically
        if index.ntotal - last_saved >= SAVE_EVERY:
            save_checkpoint()
            last_saved = index.ntotal

    if index.ntotal != last_saved:
        save_checkpoint()

    print("\nEmbedding run completed.")
    print("New chunks embedded:",
          index.ntotal - start_count)
    print("Total indexed chunks:", index.ntotal)
    print("Estimated run tokens:", estimated_run_tokens)
    print("Reported API tokens:", actual_run_tokens)

else:
    print("Embedding generation skipped.")
    print("Existing indexed chunks:", index.ntotal)


Embedding generation skipped.
Existing indexed chunks: 2808


In [110]:
def semantic_search(
    question,
    top_k=8,
    file_name=None,
    year=None,
    unique_filings=False
):

    if index.ntotal == 0:
        raise ValueError("FAISS index is empty.")

    if top_k < 1:
        raise ValueError("top_k must be positive.")

    # Helper to format retrieved passages
    def format_results(hits):

        results = []
        seen_filings = set()

        for score, idx in hits:

            chunk = ordered_chunks[int(idx)]

            filing_key = chunk["file_name"]

            if unique_filings and filing_key in seen_filings:
                continue

            seen_filings.add(filing_key)

            results.append({
                "similarity_score": round(float(score), 4),
                "chunk_id": chunk["chunk_id"],
                "file_name": chunk["file_name"],
                "deal_name": chunk["deal_name"],
                "filing_date": chunk["filing_date"],
                "filing_type": chunk["filing_type"],
                "filing_year": chunk["filing_year"],
                "doc_url": chunk["doc_url"],
                "text": chunk["text"]
            })

            if len(results) >= top_k:
                break

        return pd.DataFrame(results)

    # Search within a specific filing/year
    filtered = file_name is not None or year is not None
    if filtered:

        eligible_ids = np.array([
            i for i in range(index.ntotal)
            if (
                file_name is None
                or ordered_chunks[i]["file_name"] == file_name
            )
            and (
                year is None
                or int(ordered_chunks[i]["filing_year"]) == int(year)
            )
        ], dtype=np.int64)

        if len(eligible_ids) == 0:
            return pd.DataFrame()

    # Generate query embedding
    response = require_api_client().embeddings.create(
        model=EMBEDDING_MODEL,
        input=[question]
    )

    query_vector = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    if filtered:
        # Retrieve only eligible vectors
        vectors = index.reconstruct_batch(eligible_ids)

        # Cosine similarity (vectors are normalized)
        scores = vectors @ query_vector[0]

        ranking = np.argsort(scores)[::-1]

        hits = [
            (scores[pos], eligible_ids[pos])
            for pos in ranking
        ]

        return format_results(hits)

    # Search across all currently indexed chunks
    search_k = min(
        index.ntotal,
        max(top_k * 20, 100)
    )

    while True:
        scores, indices = index.search(
            query_vector,
            search_k
        )

        hits = [
            (score, idx)
            for score, idx in zip(scores[0], indices[0])
            if idx >= 0
        ]

        results = format_results(hits)

        if len(results) >= top_k:
            return results

        if search_k == index.ntotal:
            return results

        search_k = min(index.ntotal, search_k * 2)


In [111]:
# Verify that the entire Toyota filing is indexed
assert index.ntotal >= len(toyota_chunks), (
    "Finish embedding the Toyota filing first."
)

# Live demos are opt-in. The default uses the saved pilot artifacts.
RUN_LIVE_RETRIEVAL_DEMOS = False

if RUN_LIVE_RETRIEVAL_DEMOS:
    question = (
        "Has legal title to the receivables been "
        "transferred via a true sale at each step?"
    )

    results = semantic_search(
        question,
        top_k=8,
        file_name=TOYOTA_FILE
    )

    print("Question:", question)
    print("Matching filing:", TOYOTA_FILE)
    print("Retrieved passages:", len(results))

    for i, row in results.reset_index(drop=True).iterrows():
        print("RESULT:", i + 1)

        print("Similarity:",
              row["similarity_score"])

        print("Deal:",
              row["deal_name"])

        print("Chunk:",
              row["chunk_id"])

        print("Source:",
              row["doc_url"])

        print("\nRelevant passage:")
        print(row["text"][:1500])


Question: Has legal title to the receivables been transferred via a true sale at each step?
Matching filing: TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08_424H.htm
Retrieved passages: 8
RESULT: 1
Similarity: 0.5792
Deal: TOYOTA AUTO FINANCE RECEIVABLES LLC
Chunk: 0000929638-26-001430_988e346bba_chunk_231
Source: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm

Relevant passage:
the Warranty Receivable under the Receivables Purchase Agreement. For additional information, you should refer to “ Repurchases of Receivables ” and “ Risk Factors—Risks Primarily Related to the Bankruptcy and Insolvency of Transaction Parties and Perfection of Security Interests—The issuing entity’s interests in financed vehicles may be unenforceable or defeated ” in this prospectus. Continuity of Perfection in Financed Vehicles. Under the laws of most states, the perfected security interest in a vehicle would continue for up to four months after the vehicle is moved to a

In [112]:
test_questions = [
    "Has legal title to the receivables been transferred via a true sale at each step?",
    "Who holds the collateral and in what capacity?",
    "How are documents custodied?",
    "Are the files custodied in paper form?",
    "Is the collateral free and clear of liens?",
    "Where are collections initially deposited?",
    "How long can collections stay in a servicer account?"
]

if RUN_LIVE_RETRIEVAL_DEMOS:
    test_results = []

    for question in tqdm(
        test_questions,
        desc="Testing semantic retrieval"
    ):

        results = semantic_search(
            question,
            top_k=5,
            file_name=TOYOTA_FILE
        )

        for rank, (_, row) in enumerate(
            results.iterrows(),
            start=1
        ):

            test_results.append({
                "question": question,
                "rank": rank,
                "similarity_score": row["similarity_score"],
                "chunk_id": row["chunk_id"],
                "deal_name": row["deal_name"],
                "source": row["doc_url"],
                "passage": row["text"]
            })

    test_results_df = pd.DataFrame(test_results)

    display(test_results_df[
        ["question", "rank", "similarity_score", "chunk_id"]
    ])

else:
    test_results_df = pd.read_csv(
        processed_data / "toyota_semantic_retrieval_results.csv"
    )
    print("Loaded saved Toyota semantic retrieval:", len(test_results_df))


Testing semantic retrieval:   0%|          | 0/7 [00:00<?, ?it/s]

,question,rank,similarity_score,chunk_id
0,Has legal title to the receivables been transf...,1,0.5792,0000929638-26-001430_988e346bba_chunk_231
1,Has legal title to the receivables been transf...,2,0.5569,0000929638-26-001430_988e346bba_chunk_230
2,Has legal title to the receivables been transf...,3,0.5538,0000929638-26-001430_988e346bba_chunk_228
3,Has legal title to the receivables been transf...,4,0.5503,0000929638-26-001430_988e346bba_chunk_64
4,Has legal title to the receivables been transf...,5,0.5466,0000929638-26-001430_988e346bba_chunk_124
5,Who holds the collateral and in what capacity?,1,0.5194,0000929638-26-001430_988e346bba_chunk_204
6,Who holds the collateral and in what capacity?,2,0.5096,0000929638-26-001430_988e346bba_chunk_24
7,Who holds the collateral and in what capacity?,3,0.5029,0000929638-26-001430_988e346bba_chunk_207
8,Who holds the collateral and in what capacity?,4,0.4870,0000929638-26-001430_988e346bba_chunk_31
9,Who holds the collateral and in what capacity?,5,0.4869,0000929638-26-001430_988e346bba_chunk_205


In [113]:
retrieval_path = (
    processed_data / "toyota_semantic_retrieval_results.csv"
)

if not retrieval_path.exists():
    test_results_df.to_csv(retrieval_path, index=False)

print("Saved retrieval results to:", retrieval_path)
print("Questions tested:",
      test_results_df["question"].nunique())
print("Total retrieved passages:",
      len(test_results_df))


Saved retrieval results to: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/toyota_semantic_retrieval_results.csv
Questions tested: 7
Total retrieved passages: 35


In [114]:
# Review important Toyota chunks identified during testing
review_chunks = {
    "True sale": [62, 63, 228, 229, 239],
    "Custody": [64, 65, 207],
    "Cash collections": [208, 212]
}

toyota_lookup = {
    c["chunk_index"]: c
    for c in toyota_chunks
}

for category, chunk_numbers in review_chunks.items():
    print("CATEGORY:", category)

    for chunk_num in chunk_numbers:
        chunk = toyota_lookup[chunk_num]

        print(f"\nChunk {chunk_num}")
        print("Source:", chunk["doc_url"])
        print("Text:")
        print(chunk["text"])
        print()


CATEGORY: True sale

Chunk 62
Source: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm
Text:
information systems, including in connection with any consolidation of or change in servicing operations, could have an adverse effect on your notes. TMCC relies on its own information systems and third-party information systems to manage its operations, which creates meaningful operational risk for TMCC. Any failure or interruption of TMCC’s information systems or the third-party information systems on which it relies as a result of inadequate or failed processes or systems, human errors, employee misconduct, cyber-security attacks, catastrophic events, extreme weather conditions, security breaches, acts of vandalism, computer viruses, malware, ransomware, misplaced or lost data, or other events could disrupt TMCC’s normal operating procedures, damage its reputation and have an adverse effect on TMCC’s business, results of operations and financial conditio

In [115]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build a keyword-based index for Toyota's chunks
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

toyota_texts = [c["text"] for c in toyota_chunks]

tfidf_matrix = vectorizer.fit_transform(toyota_texts)

# Use the question we already tested
question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

# Keyword-based ranking
query_tfidf = vectorizer.transform([question])

keyword_scores = (
    tfidf_matrix @ query_tfidf.T
).toarray().ravel()

keyword_order = np.argsort(keyword_scores)[::-1][:20]

# Load saved semantic results without another API call
saved_results = pd.read_csv(
    processed_data / "toyota_semantic_retrieval_results.csv"
)

semantic_results = saved_results[
    saved_results["question"] == question
].sort_values("rank")

# Reciprocal Rank Fusion combines the two rankings
RRF_K = 60
combined_scores = {}

for rank, (_, row) in enumerate(
    semantic_results.iterrows(), start=1
):
    chunk_id = row["chunk_id"]
    combined_scores[chunk_id] = (
        combined_scores.get(chunk_id, 0)
        + 1 / (RRF_K + rank)
    )

for rank, idx in enumerate(keyword_order, start=1):
    chunk_id = toyota_chunks[idx]["chunk_id"]

    combined_scores[chunk_id] = (
        combined_scores.get(chunk_id, 0)
        + 1 / (RRF_K + rank)
    )

# Rank combined results
hybrid_ranking = sorted(
    combined_scores,
    key=combined_scores.get,
    reverse=True
)[:10]

# Inspect final ranking
chunk_lookup = {
    c["chunk_id"]: c for c in toyota_chunks
}

for rank, chunk_id in enumerate(hybrid_ranking, start=1):
    chunk = chunk_lookup[chunk_id]

    print(f"\nResult {rank}")
    print("Chunk:", chunk_id)
    print("Hybrid score:", round(combined_scores[chunk_id], 4))
    print("Preview:", chunk["text"][:500])



Result 1
Chunk: 0000929638-26-001430_988e346bba_chunk_231
Hybrid score: 0.0315
Preview: the Warranty Receivable under the Receivables Purchase Agreement. For additional information, you should refer to “ Repurchases of Receivables ” and “ Risk Factors—Risks Primarily Related to the Bankruptcy and Insolvency of Transaction Parties and Perfection of Security Interests—The issuing entity’s interests in financed vehicles may be unenforceable or defeated ” in this prospectus. Continuity of Perfection in Financed Vehicles. Under the laws of most states, the perfected security interest in

Result 2
Chunk: 0000929638-26-001430_988e346bba_chunk_230
Hybrid score: 0.0308
Preview: and the Indenture Trustee in the vehicle could be defeated through fraud or forgery by the vehicle owner or the Servicer or administrative error by state or local agencies because neither the Issuing Entity nor the Indenture Trustee will be listed as lienholder on the certificates of title. For example, the State of New

In [116]:
# Evaluate hybrid retrieval using saved semantic results
# No API calls required

def hybrid_search_cached(question, top_k=10):

    # Retrieve previously saved semantic results
    semantic_hits = saved_results[
        saved_results["question"] == question
    ].sort_values("rank")

    # Generate TF-IDF scores locally
    query_vector = vectorizer.transform([question])

    keyword_scores = (
        tfidf_matrix @ query_vector.T
    ).toarray().ravel()

    keyword_order = np.argsort(
        keyword_scores
    )[::-1]

    combined_scores = {}
    RRF_K = 60

    # Semantic ranking
    for _, row in semantic_hits.iterrows():

        chunk_id = row["chunk_id"]
        rank = int(row["rank"])

        combined_scores[chunk_id] = (
            combined_scores.get(chunk_id, 0)
            + 1 / (RRF_K + rank)
        )

    # Keyword ranking, ignoring zero-score matches
    keyword_rank = 0

    for idx in keyword_order:

        if keyword_scores[idx] <= 0:
            break

        keyword_rank += 1

        chunk_id = toyota_chunks[idx]["chunk_id"]

        combined_scores[chunk_id] = (
            combined_scores.get(chunk_id, 0)
            + 1 / (RRF_K + keyword_rank)
        )

        if keyword_rank >= 20:
            break

    # Sort final rankings
    ranked_ids = sorted(
        combined_scores,
        key=combined_scores.get,
        reverse=True
    )[:top_k]

    chunk_lookup = {
        c["chunk_id"]: c
        for c in toyota_chunks
    }

    results = []

    for rank, chunk_id in enumerate(ranked_ids, 1):

        chunk = chunk_lookup[chunk_id]

        results.append({
            "question": question,
            "rank": rank,
            "chunk_id": chunk_id,
            "hybrid_score": combined_scores[chunk_id],
            "passage": chunk["text"],
            "doc_url": chunk["doc_url"]
        })

    return pd.DataFrame(results)


# Run all seven previously tested questions
all_hybrid_results = []

for question in saved_results["question"].dropna().unique():

    results = hybrid_search_cached(
        question,
        top_k=10
    )

    all_hybrid_results.append(results)

hybrid_eval_df = pd.concat(
    all_hybrid_results,
    ignore_index=True
)

display(
    hybrid_eval_df[
        ["question", "rank", "chunk_id", "hybrid_score"]
    ]
)

print("Questions tested:",
      hybrid_eval_df["question"].nunique())

print("Total retrieved passages:",
      len(hybrid_eval_df))

,question,rank,chunk_id,hybrid_score
0,Has legal title to the receivables been transf...,1,0000929638-26-001430_988e346bba_chunk_231,0.031545
1,Has legal title to the receivables been transf...,2,0000929638-26-001430_988e346bba_chunk_230,0.030835
2,Has legal title to the receivables been transf...,3,0000929638-26-001430_988e346bba_chunk_228,0.030798
3,Has legal title to the receivables been transf...,4,0000929638-26-001430_988e346bba_chunk_64,0.030118
4,Has legal title to the receivables been transf...,5,0000929638-26-001430_988e346bba_chunk_63,0.016393
...,...,...,...,...
65,How long can collections stay in a servicer ac...,6,0000929638-26-001430_988e346bba_chunk_209,0.015625
66,How long can collections stay in a servicer ac...,7,0000929638-26-001430_988e346bba_chunk_211,0.015625
67,How long can collections stay in a servicer ac...,8,0000929638-26-001430_988e346bba_chunk_220,0.015385
68,How long can collections stay in a servicer ac...,9,0000929638-26-001430_988e346bba_chunk_215,0.015385


Questions tested: 7
Total retrieved passages: 70


In [117]:
hybrid_path = (
    processed_data / "toyota_hybrid_retrieval_results.csv"
)

hybrid_eval_df["relevance_label"] = ""
hybrid_eval_df["review_notes"] = ""

if not hybrid_path.exists():
    hybrid_eval_df.to_csv(hybrid_path, index=False)
    print("Saved:", hybrid_path)
else:
    print("Existing review file preserved:", hybrid_path)


Existing review file preserved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/toyota_hybrid_retrieval_results.csv


In [118]:
# Compare semantic and hybrid rankings
# No API calls

semantic_ranks = saved_results[
    ["question", "chunk_id", "rank"]
].rename(columns={"rank": "semantic_rank"})

hybrid_ranks = hybrid_eval_df[
    ["question", "chunk_id", "rank", "hybrid_score"]
].rename(columns={"rank": "hybrid_rank"})

comparison_df = pd.merge(
    semantic_ranks,
    hybrid_ranks,
    on=["question", "chunk_id"],
    how="outer"
)

# Attach the original prospectus passages
chunk_lookup = {
    c["chunk_id"]: c
    for c in toyota_chunks
}

comparison_df["passage"] = comparison_df[
    "chunk_id"
].map(lambda x: chunk_lookup[x]["text"])

comparison_df["relevance_label"] = ""
comparison_df["review_notes"] = ""

comparison_df = comparison_df.sort_values(
    ["question", "hybrid_rank"],
    na_position="last"
)

review_path = (
    processed_data / "toyota_retrieval_comparison.csv"
)

if not review_path.exists():
    comparison_df.to_csv(review_path, index=False)
    print("Saved:", review_path)
else:
    print("Existing review file preserved.")

print("Questions:", comparison_df["question"].nunique())
print("Passages to review:", len(comparison_df))

display(comparison_df[
    ["question", "chunk_id",
     "semantic_rank", "hybrid_rank"]
].head(20))


Existing review file preserved.
Questions: 7
Passages to review: 70


,question,chunk_id,semantic_rank,hybrid_rank
3,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_207,2.0,1
1,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_187,4.0,2
6,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_266,3.0,3
9,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_65,1.0,4
7,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_279,NaN,5
4,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_214,NaN,6
5,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_220,NaN,7
2,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_190,5.0,8
8,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_64,NaN,9
0,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_121,NaN,10


In [119]:
question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

review = comparison_df[
    comparison_df["question"] == question
].sort_values("hybrid_rank")

for _, row in review.head(10).iterrows():

    print("Chunk:", row["chunk_id"])
    print("Semantic rank:", row["semantic_rank"])
    print("Hybrid rank:", row["hybrid_rank"])
    print("\nPassage:")
    print(row["passage"])


Chunk: 0000929638-26-001430_988e346bba_chunk_231
Semantic rank: 1.0
Hybrid rank: 1

Passage:
the Warranty Receivable under the Receivables Purchase Agreement. For additional information, you should refer to “ Repurchases of Receivables ” and “ Risk Factors—Risks Primarily Related to the Bankruptcy and Insolvency of Transaction Parties and Perfection of Security Interests—The issuing entity’s interests in financed vehicles may be unenforceable or defeated ” in this prospectus. Continuity of Perfection in Financed Vehicles. Under the laws of most states, the perfected security interest in a vehicle would continue for up to four months after the vehicle is moved to and re-registered by its owner in a state that is different from the one in which it is initially registered. A majority of states generally require surrender of a certificate of title to re-register a vehicle. In those states that require a secured party to hold possession of the certificate of title to maintain perfection of 

In [120]:
# Uses existing results; no API calls

question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

# Provisional relevance labels from passage review
# 2 = directly relevant
# 1 = partially relevant
# 0 = not directly relevant

labels = {
    231: 0,
    230: 0,
    228: 2,
    64: 1,
    63: 2,
    239: 2,
    73: 1,
    229: 2,
    124: 1,
    62: 2
}

review = comparison_df[
    comparison_df["question"] == question
].copy()

review["chunk_number"] = (
    review["chunk_id"]
    .str.extract(r"_chunk_(\d+)$")
    .astype(int)
)

review["relevance"] = review[
    "chunk_number"
].map(labels)

assert review["relevance"].notna().all(), (
    "Some retrieved chunks still need relevance labels."
)

for method, rank_column in [
    ("Semantic", "semantic_rank"),
    ("Hybrid", "hybrid_rank")
]:
    top5 = (
        review.dropna(subset=[rank_column])
        .sort_values(rank_column)
        .head(5)
    )

    direct_hits = (top5["relevance"] == 2).sum()
    precision = direct_hits / len(top5)

    print(f"{method} Precision@5: {precision:.2f}")
    print(f"Direct evidence hits: {direct_hits}/5\n")


Semantic Precision@5: 0.20
Direct evidence hits: 1/5

Hybrid Precision@5: 0.40
Direct evidence hits: 2/5



In [121]:
from collections import Counter

# Count chunks per filing
total_per_file = Counter(
    c["file_name"] for c in ordered_chunks
)

indexed_per_file = Counter(
    c["file_name"]
    for c in ordered_chunks[:index.ntotal]
)

coverage = []

for file_name, total in total_per_file.items():
    indexed = indexed_per_file[file_name]

    coverage.append({
        "file_name": file_name,
        "indexed_chunks": indexed,
        "total_chunks": total,
        "coverage_pct": round(indexed / total * 100, 2),
        "fully_indexed": indexed == total
    })

coverage_df = pd.DataFrame(coverage)

print("Total filings:", len(coverage_df))
print("Fully indexed filings:",
      coverage_df["fully_indexed"].sum())

print("Partially indexed filings:",
      ((coverage_df["indexed_chunks"] > 0) &
       (~coverage_df["fully_indexed"])).sum())

print("Filings not yet indexed:",
      (coverage_df["indexed_chunks"] == 0).sum())

display(
    coverage_df[
        coverage_df["indexed_chunks"] > 0
    ].sort_values(
        "coverage_pct", ascending=False
    )
)


Total filings: 122
Fully indexed filings: 12
Partially indexed filings: 1
Filings not yet indexed: 109


,file_name,indexed_chunks,total_chunks,coverage_pct,fully_indexed
0,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08...,312,312,100.00,True
1,AFS_SENSUB_CORP__2024-05-15_424H.htm,206,206,100.00,True
2,AFS_SENSUB_CORP__2024-06-26_424H.htm,214,214,100.00,True
3,AFS_SENSUB_CORP__2024-10-02_424H.htm,214,214,100.00,True
4,AFS_SENSUB_CORP__2025-01-06_424H.htm,211,211,100.00,True
5,AFS_SENSUB_CORP__2025-05-01_424H.htm,226,226,100.00,True
6,AFS_SENSUB_CORP__2025-07-03_424H.htm,217,217,100.00,True
7,AFS_SENSUB_CORP__2025-10-23_424H.htm,199,199,100.00,True
8,AFS_SENSUB_CORP__2026-01-05_424H.htm,211,211,100.00,True
9,AFS_SENSUB_CORP__2026-04-03_424H.htm,196,196,100.00,True


In [122]:
from prospectus_backend import build_keyword_index

# Only the chunks with saved embeddings
indexed_chunks = ordered_chunks[:index.ntotal]

# Exclude partially indexed documents by default
complete_files = set(
    coverage_df.loc[
        coverage_df["fully_indexed"], "file_name"
    ]
)

# Create TF-IDF index for keyword retrieval
global_vectorizer, global_tfidf = build_keyword_index(indexed_chunks)

# Reuse saved OpenAI vectors
indexed_vectors = index.reconstruct_n(
    0, index.ntotal
)

print("Indexed chunks:", len(indexed_chunks))
print("Fully indexed filings:", len(complete_files))
print("TF-IDF shape:", global_tfidf.shape)
print("Embedding matrix:", indexed_vectors.shape)

Indexed chunks: 2808
Fully indexed filings: 12
TF-IDF shape: (2808, 125136)
Embedding matrix: (2808, 1536)


In [123]:
from prospectus_backend import hybrid_search as shared_hybrid_search


def hybrid_search(question, top_k=8, file_name=None, year=None,
                  unique_filings=False, include_partial=False):
    return shared_hybrid_search(
        question, top_k, file_name, year, unique_filings, include_partial,
        indexed_chunks=indexed_chunks, complete_files=complete_files,
        indexed_vectors=indexed_vectors, global_vectorizer=global_vectorizer,
        global_tfidf=global_tfidf, require_api_client=require_api_client,
    )


In [124]:
question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

if RUN_LIVE_RETRIEVAL_DEMOS:
    toyota_results = hybrid_search(
        question,
        file_name=TOYOTA_FILE,
        top_k=10
    )

else:
    saved_pilot = pd.read_csv(
        processed_data / "milestone2_task3_hybrid_pilot.csv",
        keep_default_na=False
    )
    toyota_results = saved_pilot.loc[
        saved_pilot["search_type"] == "deal_specific"
    ].copy()

display(
    toyota_results[
        ["chunk_id", "hybrid_score"]
    ]
)

,chunk_id,hybrid_score
0,0000929638-26-001430_988e346bba_chunk_231,0.031319
1,0000929638-26-001430_988e346bba_chunk_228,0.031025
2,0000929638-26-001430_988e346bba_chunk_230,0.030622
3,0000929638-26-001430_988e346bba_chunk_229,0.030536
4,0000929638-26-001430_988e346bba_chunk_64,0.029911
5,0000929638-26-001430_988e346bba_chunk_63,0.028950
6,0000929638-26-001430_988e346bba_chunk_235,0.028595
7,0000929638-26-001430_988e346bba_chunk_239,0.027505
8,0000929638-26-001430_988e346bba_chunk_74,0.027364
9,0000929638-26-001430_988e346bba_chunk_125,0.026857


In [125]:
cross_deal_question = (
    "Which deals describe third-party custody "
    "or electronic control of receivables?"
)

if RUN_LIVE_RETRIEVAL_DEMOS:
    cross_deal_results = hybrid_search(
        cross_deal_question,
        top_k=8,
        unique_filings=True
    )

else:
    saved_pilot = pd.read_csv(
        processed_data / "milestone2_task3_hybrid_pilot.csv",
        keep_default_na=False
    )
    cross_deal_results = saved_pilot.loc[
        saved_pilot["search_type"] == "cross_deal"
    ].copy()

display(
    cross_deal_results[
        [
            "deal_name",
            "filing_date",
            "hybrid_score",
            "chunk_id"
        ]
    ]
)

# Inspect supporting evidence
for _, row in cross_deal_results.iterrows():
    print("Deal:", row["deal_name"])
    print("Date:", row["filing_date"])
    print("Source:", row["doc_url"])
    print("Passage:", row["text"][:1000])

,deal_name,filing_date,hybrid_score,chunk_id
0,TOYOTA AUTO FINANCE RECEIVABLES LLC,2026-04-08,0.032522,0000929638-26-001430_988e346bba_chunk_65
1,AMERICAN EXPRESS CREDIT ACCOUNT MASTER TRUST,2024-04-11,0.026199,0001193125-24-092834_606c3ec708_chunk_40
2,AFS SENSUB CORP,2025-01-06,0.015385,0001193125-25-002125_5bcf3f4681_chunk_145
3,AFS SENSUB CORP,2024-05-15,0.014925,0001193125-24-139394_6460085a42_chunk_146
4,AFS SENSUB CORP,2026-04-03,0.014706,0001193125-26-142046_331464f52f_chunk_129
5,AFS SENSUB CORP,2024-10-02,0.014493,0001193125-24-231284_dc3a6dab17_chunk_144
6,AFS SENSUB CORP,2024-06-26,0.014085,0001193125-24-169068_b2ca7363dd_chunk_145
7,AFS SENSUB CORP,2026-01-05,0.013889,0001193125-26-002340_ed1eab1425_chunk_145


Deal: TOYOTA AUTO FINANCE RECEIVABLES LLC
Date: 2026-04-08
Source: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm
Passage: contract if (a) there is a “single authoritative copy” of the electronic contract that is readily distinguishable from all other copies and which identifies TMCC as the owner, (b) all other copies of the electronic contract indicate that they are not the “authoritative copy” of the electronic contract, (c) any revisions to the authoritative copy of the electronic contract are readily identifiable as either authorized or unauthorized revisions, and (d) authorized revisions of the electronic contract cannot be made without TMCC’s participation. However, it is possible that another person could acquire an interest in an electronic contract that is superior to TMCC’s interest (and accordingly, the issuing entity’s interest). This could occur if TMCC ceases to have “control” over the electronic contract that is maintained by TMCC 

In [126]:
pilot_results = pd.concat(
    [
        toyota_results.assign(
            question=question,
            search_type="deal_specific"
        ),
        cross_deal_results.assign(
            question=cross_deal_question,
            search_type="cross_deal"
        )
    ],
    ignore_index=True
)

output_path = (
    processed_data /
    "milestone2_task3_hybrid_pilot.csv"
)

if not output_path.exists():
    pilot_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Total results:", len(pilot_results))


Saved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/milestone2_task3_hybrid_pilot.csv
Total results: 18


In [127]:
review_df = cross_deal_results.copy().reset_index(drop=True)

review_df.insert(
    0, "rank",
    range(1, len(review_df) + 1)
)

# Labels to fill in after inspecting the full passages:
# 2 = directly relevant
# 1 = partially relevant
# 0 = irrelevant

review_df["relevance_label"] = pd.NA
review_df["review_notes"] = ""

review_path = (
    processed_data /
    "cross_deal_retrieval_review.csv"
)

if not review_path.exists():
    review_df.to_csv(review_path, index=False)
    print("Review template saved:", review_path)
else:
    print("Existing review file preserved:", review_path)

# Print full passages for manual review
for _, row in review_df.iterrows():
    print("RANK:", row["rank"])
    print("DEAL:", row["deal_name"])
    print("DATE:", row["filing_date"])
    print("CHUNK:", row["chunk_id"])
    print("SOURCE:", row["doc_url"])
    print("\nFULL PASSAGE:")
    print(row["text"])


Existing review file preserved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/cross_deal_retrieval_review.csv
RANK: 1
DEAL: TOYOTA AUTO FINANCE RECEIVABLES LLC
DATE: 2026-04-08
CHUNK: 0000929638-26-001430_988e346bba_chunk_65
SOURCE: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm

FULL PASSAGE:
contract if (a) there is a “single authoritative copy” of the electronic contract that is readily distinguishable from all other copies and which identifies TMCC as the owner, (b) all other copies of the electronic contract indicate that they are not the “authoritative copy” of the electronic contract, (c) any revisions to the authoritative copy of the electronic contract are readily identifiable as either authorized or unauthorized revisions, and (d) authorized revisions of the electronic contract cannot be made without TMCC’s participation. However, it is possible that another person coul

In [128]:
evaluation_path = (
    processed_data /
    "cross_deal_retrieval_labeled.csv"
)

# Historical labels belong to the saved chunks, regardless of live ranks.
cross_review = pd.read_csv(evaluation_path)

def precision_at_k(df, k):
    top_k = df.sort_values("rank").head(k)

    relevant = (
        top_k["relevance_label"] == 2
    ).sum()

    return relevant / len(top_k)

print("Cross-deal retrieval evaluation")

print("Precision@5:",
      round(precision_at_k(cross_review, 5), 3))

print("Precision@8:",
      round(precision_at_k(cross_review, 8), 3))

print("\nRead saved labels:", evaluation_path)

display(
    cross_review[
        ["rank", "deal_name", "filing_date",
         "relevance_label", "chunk_id"]
    ]
)

Cross-deal retrieval evaluation
Precision@5: 0.8
Precision@8: 0.875

Saved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/cross_deal_retrieval_labeled.csv


,rank,deal_name,filing_date,relevance_label,chunk_id
0,1,TOYOTA AUTO FINANCE RECEIVABLES LLC,2026-04-08,2,0000929638-26-001430_988e346bba_chunk_65
1,2,AMERICAN EXPRESS CREDIT ACCOUNT MASTER TRUST,2024-04-11,0,0001193125-24-092834_606c3ec708_chunk_40
2,3,AFS SENSUB CORP,2025-01-06,2,0001193125-25-002125_5bcf3f4681_chunk_145
3,4,AFS SENSUB CORP,2024-05-15,2,0001193125-24-139394_6460085a42_chunk_146
4,5,AFS SENSUB CORP,2026-04-03,2,0001193125-26-142046_331464f52f_chunk_129
5,6,AFS SENSUB CORP,2024-10-02,2,0001193125-24-231284_dc3a6dab17_chunk_144
6,7,AFS SENSUB CORP,2024-06-26,2,0001193125-24-169068_b2ca7363dd_chunk_145
7,8,AFS SENSUB CORP,2026-01-05,2,0001193125-26-002340_ed1eab1425_chunk_145


### Task 2

In [129]:
import json
import re
import hashlib
from pathlib import Path
import pandas as pd

LLM_MODEL = "gpt-5.6-luna"

# Limit evidence per question to control token usage
TOP_K = 8
MAX_COMPLETION_TOKENS = 1800

PROMPT_VERSION = "task2_v2"

TOYOTA_FILE = (
    "TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_"
    "2026-04-08_424H.htm"
)

# Check that Task 3 is loaded
assert "client" in globals(), "Load your OpenAI client first."
assert "index" in globals(), "Load your FAISS index first."
assert "ordered_chunks" in globals(), "Load Task 3 chunks first."
assert "coverage_df" in globals(), "Run the coverage cell first."
assert "processed_data" in globals(), "processed_data is missing."

processed_data = Path(processed_data)

TASK2_DIR = processed_data / "task2_answers"
TASK2_DIR.mkdir(parents=True, exist_ok=True)

# Saved retrieval results from Task 3
pilot_path = (processed_data / "milestone2_task3_hybrid_pilot.csv")

toyota_hybrid_path = (processed_data / "toyota_hybrid_retrieval_results.csv")

assert pilot_path.exists(), (f"Missing saved pilot results: {pilot_path}")

assert toyota_hybrid_path.exists(), (f"Missing saved Toyota hybrid results: {toyota_hybrid_path}")

pilot_df = pd.read_csv(pilot_path)
toyota_hybrid_df = pd.read_csv(toyota_hybrid_path)

# FAISS row i corresponds to ordered_chunks[i]
indexed_chunks = ordered_chunks[:index.ntotal]

indexed_lookup = {
    chunk["chunk_id"]: chunk
    for chunk in indexed_chunks
}

assert len(indexed_lookup) == index.ntotal, ("Duplicate chunk IDs or an inconsistent index mapping.")

# Only use filings with complete indexing for this pilot
complete_files = set(
    coverage_df.loc[
        coverage_df["fully_indexed"], "file_name"
    ]
)

print("Task 2 configuration ready.")
print("Available indexed chunks:", index.ntotal)
print("Fully indexed filings:", len(complete_files))
print("Task 2 output folder:", TASK2_DIR)


Task 2 configuration ready.
Available indexed chunks: 2808
Fully indexed filings: 12
Task 2 output folder: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/task2_answers


In [130]:
def find_saved_question(questions, required_phrases):
    matches = [
        q for q in questions
        if all(
            phrase in q.lower()
            for phrase in required_phrases
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one question containing "
            f"{required_phrases}, found: {matches}"
        )

    return matches[0]

toyota_questions = (
    toyota_hybrid_df["question"]
    .dropna()
    .unique()
    .tolist()
)

TRUE_SALE_QUESTION = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

COLLECTION_QUESTION = find_saved_question(
    toyota_questions,
    ["how long", "collection"]
)

PAPER_QUESTION = find_saved_question(
    toyota_questions,
    ["paper form"]
)

CROSS_DEAL_QUESTION = (
    "Which deals describe third-party custody "
    "or electronic control of receivables?"
)

# This question tests whether the LLM admits when the provided evidence lacks a specific fact
ABSTENTION_QUESTION = (
    "What is the exact name and street address "
    "of the third-party custodian responsible for "
    "Toyota's electronic loan contracts?"
)

TEST_CASES = [
    {
        "id": "true_sale",
        "question": TRUE_SALE_QUESTION,
        "retrieval_question": TRUE_SALE_QUESTION,
        "retrieval_mode": "toyota_pilot",
        "scope": "Toyota filing only"
    },
    {
        "id": "collections",
        "question": COLLECTION_QUESTION,
        "retrieval_question": COLLECTION_QUESTION,
        "retrieval_mode": "toyota_hybrid",
        "scope": "Toyota filing only"
    },
    {
        "id": "paper_custody",
        "question": PAPER_QUESTION,
        "retrieval_question": PAPER_QUESTION,
        "retrieval_mode": "toyota_hybrid",
        "scope": "Toyota filing only"
    },
    {
        "id": "cross_deal_custody",
        "question": CROSS_DEAL_QUESTION,
        "retrieval_question": CROSS_DEAL_QUESTION,
        "retrieval_mode": "cross_deal_pilot",
        "scope": (
            "Only the fully indexed prospectuses "
            "included in Task 3"
        )
    },
    {
        "id": "insufficient_evidence",
        "question": ABSTENTION_QUESTION,
        # Reuse custody evidence to test answer-stage abstention without another embedding request
        "retrieval_question": PAPER_QUESTION,
        "retrieval_mode": "toyota_hybrid",
        "scope": (
            "Toyota filing; fixed-context abstention test"
        )
    }
]

for i, case in enumerate(TEST_CASES, 1):
    print(f"\nTEST {i}: {case['id']}")
    print("Question:", case["question"])
    print("Scope:", case["scope"])



TEST 1: true_sale
Question: Has legal title to the receivables been transferred via a true sale at each step?
Scope: Toyota filing only

TEST 2: collections
Question: How long can collections stay in a servicer account?
Scope: Toyota filing only

TEST 3: paper_custody
Question: Are the files custodied in paper form?
Scope: Toyota filing only

TEST 4: cross_deal_custody
Question: Which deals describe third-party custody or electronic control of receivables?
Scope: Only the fully indexed prospectuses included in Task 3

TEST 5: insufficient_evidence
Question: What is the exact name and street address of the third-party custodian responsible for Toyota's electronic loan contracts?
Scope: Toyota filing; fixed-context abstention test


In [131]:
def prepare_evidence(case, top_k=TOP_K):
    question = case["retrieval_question"]
    mode = case["retrieval_mode"]

    if mode == "toyota_pilot":
        rows = pilot_df[
            (pilot_df["question"] == question)
            & (pilot_df["search_type"] == "deal_specific")
            & (pilot_df["file_name"] == TOYOTA_FILE)
        ].copy()

    elif mode == "toyota_hybrid":
        rows = toyota_hybrid_df[toyota_hybrid_df["question"] == question].copy()

    elif mode == "cross_deal_pilot":
        rows = pilot_df[(pilot_df["question"] == question) & (pilot_df["search_type"] == "cross_deal")].copy()

    else:
        raise ValueError(f"Unknown retrieval mode: {mode}")

    if rows.empty:
        raise ValueError(
            f"No saved retrieval results for: {question}"
        )

    # Sort if rank is available; otherwise preserve the saved result order
    if "rank" in rows.columns:
        rows = rows.sort_values("rank")

    rows = (
        rows.drop_duplicates(subset="chunk_id")
        .head(top_k)
        .reset_index(drop=True)
    )

    evidence = []

    for rank, (_, row) in enumerate(rows.iterrows(), 1):
        chunk_id = row["chunk_id"]

        if chunk_id not in indexed_lookup:
            raise ValueError(
                f"Chunk missing from current index: {chunk_id}"
            )

        chunk = indexed_lookup[chunk_id]

        if chunk["file_name"] not in complete_files:
            raise ValueError(
                "Retrieval result belongs to a partially "
                f"indexed filing: {chunk['file_name']}"
            )

        if mode.startswith("toyota"):
            assert chunk["file_name"] == TOYOTA_FILE

        evidence.append({
            "source_ref": f"S{rank}",
            "rank": rank,
            "chunk_id": chunk_id,
            "file_name": chunk["file_name"],
            "deal_name": chunk["deal_name"],
            "filing_date": chunk["filing_date"],
            "doc_url": chunk["doc_url"],
            "text": chunk["text"]
        })

    return pd.DataFrame(evidence)


# Prepare and cache all five sets of evidence locally
TEST_EVIDENCE = {}

for case in TEST_CASES:
    evidence = prepare_evidence(case)
    TEST_EVIDENCE[case["id"]] = evidence

    print(
        f"{case['id']}: "
        f"{len(evidence)} passages, "
        f"{evidence['file_name'].nunique()} filing(s)"
    )

# Check important Toyota evidence
true_sale_ids = TEST_EVIDENCE["true_sale"]["chunk_id"]
collection_ids = TEST_EVIDENCE["collections"]["chunk_id"]

assert true_sale_ids.str.endswith("_chunk_239").any(), ("Important true-sale passage 239 is missing.")

assert collection_ids.str.endswith("_chunk_212").any(), ("Important collections passage 212 is missing.")

print("\nImportant supporting passages are included.")
print("Evidence preparation complete.")

true_sale: 8 passages, 1 filing(s)
collections: 8 passages, 1 filing(s)
paper_custody: 8 passages, 1 filing(s)
cross_deal_custody: 8 passages, 8 filing(s)
insufficient_evidence: 8 passages, 1 filing(s)

Important supporting passages are included.
Evidence preparation complete.


In [132]:
from prospectus_backend import SYSTEM_PROMPT, generate_answer_from_passages as shared_generate_answer


def generate_answer_from_passages(question, passages, scope="Supplied prospectus passages"):
    return shared_generate_answer(
        question, passages, scope, require_api_client=require_api_client,
        system_prompt=SYSTEM_PROMPT, model=LLM_MODEL, prompt_version=PROMPT_VERSION,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )


print("LLM answer-generation function ready.")
print("Model:", LLM_MODEL)
print("Prompt version:", PROMPT_VERSION)


LLM answer-generation function ready.
Model: gpt-5.6-luna
Prompt version: task2_v2


In [133]:
RUN_NEW_LLM_CALLS = False

# Maximum new LLM requests in this execution
MAX_NEW_LLM_CALLS = 4

# Existing outputs from earlier Task 2 work
LEGACY_ANSWERS = {
    "true_sale": (
        processed_data / "toyota_true_sale_rag_answer.json"
    ),
    "collections": (
        processed_data / "toyota_collections_rag_answer.json"
    )
}

TASK2_RESULTS = {}
new_call_count = 0

for case in TEST_CASES:
    case_id = case["id"]
    answer_path = (
        TASK2_DIR / f"{case_id}_answer.json"
    )

    print("\n" + "=" * 70)
    print("TEST:", case_id)
    print("QUESTION:", case["question"])

    result = None

    # Prefer the Task 2 cached result
    if answer_path.exists():
        result = json.loads(
            answer_path.read_text(encoding="utf-8")
        )

        print("Loaded cached Task 2 answer.")

    # Reuse earlier saved answers where possible
    elif (
        case_id in LEGACY_ANSWERS
        and LEGACY_ANSWERS[case_id].exists()
    ):

        result = json.loads(
            LEGACY_ANSWERS[case_id].read_text(
                encoding="utf-8"
            )
        )

        print("Loaded existing baseline answer.")

    # Otherwise generate one new answer
    elif RUN_NEW_LLM_CALLS:
        if new_call_count >= MAX_NEW_LLM_CALLS:
            print("API call limit reached. Skipping.")
            continue

        print("Generating new answer...")

        # If the API request fails, no incomplete answer will be cached. Earlier completed answers remain saved.
        result = generate_answer_from_passages(
            question=case["question"],
            passages=TEST_EVIDENCE[case_id],
            scope=case["scope"]
        )

        new_call_count += 1

        answer_path.write_text(
            json.dumps(result, indent=2),
            encoding="utf-8"
        )

        print("Saved:", answer_path)

    else:
        print(
            "Not run. Set RUN_NEW_LLM_CALLS = True."
        )
        continue

    TASK2_RESULTS[case_id] = result

    print("\nANSWER:\n")
    print(result["answer"])

    print("\nCITATIONS USED:")
    print(result.get("citations_used", []))

    print("\nUNKNOWN CITATIONS:")
    print(result.get("unknown_citations", []))

    usage = result.get("usage") or {}

    print("\nTOTAL TOKENS:")
    print(usage.get("total_tokens", "Unavailable"))

print("TASK 2 TEST SUMMARY: ")
print("Answers available:", len(TASK2_RESULTS))
print("New LLM calls this run:", new_call_count)



TEST: true_sale
QUESTION: Has legal title to the receivables been transferred via a true sale at each step?
Loaded existing baseline answer.

ANSWER:

Conclusion:

No. The passages do not establish that legal title was transferred via a legally effective “true sale” at each step. They describe the intended structure as:

1. TMCC sells and assigns the Receivables to the Depositor;
2. the Depositor sells and assigns them to the Issuing Entity; and
3. the Issuing Entity pledges its interest to the Indenture Trustee.

The first two transfers are intended to be sales, but the prospectus expressly acknowledges that a court could recharacterize either transfer as a pledge or security interest rather than a true sale. The transfer to the Indenture Trustee is expressly a pledge, not a sale. [S2][S4][S6][S7][S8]

Supporting Evidence:

- Under the Receivables Purchase Agreement, TMCC “will sell and assign” its interest in the Receivables to the Depositor. Under the Sale and Servicing Agreement, 

In [134]:
evaluation_rows = []

for case in TEST_CASES:
    case_id = case["id"]
    if case_id not in TASK2_RESULTS:
        continue

    result = TASK2_RESULTS[case_id]
    answer = result["answer"]
    sources = result.get("sources", [])

    known_refs = {
        s["reference"] for s in sources
    }

    cited_refs = set(
        re.findall(r"\[(S\d+)\]", answer)
    )

    unknown_refs = cited_refs - known_refs
    usage = result.get("usage") or {}

    evaluation_rows.append({
        "test_id": case_id,
        "question": case["question"],
        "model": result.get("model"),
        "source_count": len(sources),
        "citations_used": len(cited_refs),
        "has_citations": len(cited_refs) > 0,
        "valid_citation_ids": bool(cited_refs) and not unknown_refs,
        "finish_reason": result.get("finish_reason"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get(
            "completion_tokens"
        ),
        "total_tokens": usage.get("total_tokens"),
        "factually_correct": "",
        "citations_support_claims": "",
        "handles_uncertainty": "",
        "review_notes": ""
    })

task2_eval_df = pd.DataFrame(evaluation_rows)

display(task2_eval_df)

# Save without overwriting previous manual reviews
task2_eval_path = (
    TASK2_DIR / "task2_answer_evaluation.csv"
)

if not task2_eval_path.exists():
    task2_eval_df.to_csv(
        task2_eval_path,
        index=False
    )

    print("\nEvaluation saved:", task2_eval_path)

else:
    print(
        "\nExisting evaluation file preserved:",
        task2_eval_path
    )

print("\nEvaluation summary:")

if not task2_eval_df.empty:
    print(
        "Answers evaluated:",
        len(task2_eval_df)
    )

    print(
        "Answers with valid citation IDs:",
        int(
            task2_eval_df["valid_citation_ids"].sum()
        )
    )

    print(
        "Answers containing citations:",
        int(
            task2_eval_df["has_citations"].sum()
        )
    )

,test_id,question,model,source_count,citations_used,has_citations,valid_citation_ids,finish_reason,prompt_tokens,completion_tokens,total_tokens,factually_correct,citations_support_claims,handles_uncertainty,review_notes
0,true_sale,Has legal title to the receivables been transf...,gpt-5.6-luna,8,7,True,True,None,5899,785,6684,,,,
1,collections,How long can collections stay in a servicer ac...,gpt-5.6-luna,8,2,True,True,stop,6310,491,6801,,,,
2,paper_custody,Are the files custodied in paper form?,gpt-5.6-luna,8,2,True,True,stop,6459,298,6757,,,,
3,cross_deal_custody,Which deals describe third-party custody or el...,gpt-5.6-luna,8,8,True,True,stop,5941,1215,7156,,,,
4,insufficient_evidence,What is the exact name and street address of t...,gpt-5.6-luna,8,2,True,True,stop,6477,214,6691,,,,



Existing evaluation file preserved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/task2_answers/task2_answer_evaluation.csv

Evaluation summary:
Answers evaluated: 5
Answers with valid citation IDs: 5
Answers containing citations: 5


In [135]:
from prospectus_backend import (
    answer_new_question as shared_answer_question,
    question_cache_path as shared_question_cache_path,
)


def question_cache_path(question, file_name, top_k):
    return shared_question_cache_path(
        question, file_name, top_k, cache_dir=TASK2_DIR, order_hash=order_hash,
        source_hash=source_hash, indexed_count=index.ntotal,
        model=LLM_MODEL, prompt_version=PROMPT_VERSION,
    )


def answer_new_question(question, file_name=None, top_k=TOP_K, run_api=False):
    return shared_answer_question(
        question, file_name, top_k, run_api,
        cache_path=question_cache_path(question, file_name, top_k),
        hybrid_search=hybrid_search, indexed_lookup=indexed_lookup,
        generate_answer_from_passages=generate_answer_from_passages,
    )


In [136]:
# Example: Query a specific Toyota filing
# Keep run_api=False until you want to spend API credits.
example_question = (
    "Who is responsible for maintaining custody "
    "of the receivable files?"
)

example_result = answer_new_question(
    question=example_question,
    file_name=TOYOTA_FILE,
    top_k=8,
    run_api=False
)

if example_result is not None:
    print(example_result["answer"])

New question requires one query-embedding request and one LLM request.
Set run_api=True to execute.


In [137]:
manual_reviews = {
    "true_sale": {
        "factually_correct": "Needs minor revision",
        "citations_support_claims": "Mostly",
        "handles_uncertainty": "Needs improvement",
        "review_notes": (
            "Correct transfer chain and legal risks, "
            "but opening 'No' is too definitive. "
            "Chunk 235 contains a possible source inconsistency."
        )
    },
    "collections": {
        "factually_correct": "Pass",
        "citations_support_claims": "Pass",
        "handles_uncertainty": "Pass",
        "review_notes": (
            "Correctly explains the conditional Payment Date "
            "and two-business-day deposit requirements."
        )
    },
    "paper_custody": {
        "factually_correct": "Pass",
        "citations_support_claims": "Pass",
        "handles_uncertainty": "Pass",
        "review_notes": (
            "Correctly distinguishes possession of tangible "
            "contracts from control of electronic contracts."
        )
    },
    "cross_deal_custody": {
        "factually_correct": "Pass",
        "citations_support_claims": "Pass",
        "handles_uncertainty": "Pass",
        "review_notes": (
            "Identifies seven supported filings and correctly "
            "does not treat the American Express excerpt "
            "as evidence of electronic custody."
        )
    },
    "insufficient_evidence": {
        "factually_correct": "Pass",
        "citations_support_claims": "Pass",
        "handles_uncertainty": "Pass",
        "review_notes": (
            "Correctly abstains from inventing a custodian "
            "name or physical address."
        )
    }
}

reviewed_df = task2_eval_df.copy()

for test_id, review in manual_reviews.items():
    mask = reviewed_df["test_id"] == test_id
    for column, value in review.items():
        reviewed_df.loc[mask, column] = value

# Save separately to preserve the original evaluation CSV
reviewed_path = (
    TASK2_DIR / "task2_manual_evaluation.csv"
)

if not reviewed_path.exists():
    reviewed_df.to_csv(reviewed_path, index=False)
else:
    reviewed_df = pd.read_csv(reviewed_path, keep_default_na=False)
print("Manual evaluation saved:", reviewed_path)

display(
    reviewed_df[
        [
            "test_id",
            "factually_correct",
            "citations_support_claims",
            "handles_uncertainty",
            "review_notes"
        ]
    ]
)

Manual evaluation saved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/task2_answers/task2_manual_evaluation.csv


,test_id,factually_correct,citations_support_claims,handles_uncertainty,review_notes
0,true_sale,Needs minor revision,Mostly,Needs improvement,"Correct transfer chain and legal risks, but op..."
1,collections,Pass,Pass,Pass,Correctly explains the conditional Payment Dat...
2,paper_custody,Pass,Pass,Pass,Correctly distinguishes possession of tangible...
3,cross_deal_custody,Pass,Pass,Pass,Identifies seven supported filings and correct...
4,insufficient_evidence,Pass,Pass,Pass,Correctly abstains from inventing a custodian ...


In [138]:
import pandas as pd
import numpy as np
import json
import re
import hashlib
from pathlib import Path

# Find the CSV in the repository or notebook directory
question_locations = [
    PROJECT_ROOT / "test_questions.csv",
    PROJECT_ROOT / "eval" / "test_questions.csv",
    Path.cwd() / "test_questions.csv",
    Path.cwd().parent / "test_questions.csv"
]

question_path = next(
    (p for p in question_locations if p.exists()),
    None
)

if question_path is None:
    raise FileNotFoundError(
        "Place test_questions.csv in the project root "
        "or the data folder."
    )

questions_df = pd.read_csv(
    question_path,
    keep_default_na=False
)

assert set(questions_df["id"]) == {f"q{i:02d}" for i in range(1, 21)}
assert len(questions_df) == 20 and questions_df["question"].str.strip().ne("").all()
assert {"id", "question", "expected_issuers"}.issubset(
    questions_df.columns
)

EVAL_DIR = processed_data / "m2_evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Map actual SEC filenames to the issuer labels in the CSV
ISSUER_PREFIXES = {
    "TOYOTA": "TOYOTA_AUTO_FINANCE",
    "AFS SENSUB": "AFS_SENSUB",
    "AMERICAN EXPRESS": "AMERICAN_EXPRESS",
    "AMERICAN HONDA": "AMERICAN_HONDA",
    "CARMAX": "CARMAX",
    "FORD CREDIT": "FORD_CREDIT",
    "BRIDGECREST": "BRIDGECREST",
    "SANTANDER DRIVE": "SANTANDER_DRIVE",
    "WORLD OMNI": "WORLD_OMNI",
    "HYUNDAI": "HYUNDAI",
    "ALLY": "ALLY_AUTO_ASSETS",
    "HARLEY-DAVIDSON": "HARLEY_DAVIDSON",
    "BMW AUTO LEASING": "BMW_AUTO_LEASING",
    "VERIZON": "VERIZON",
    "WELLS FARGO COMMERCIAL MORTGAGE":
        "WELLS_FARGO_COMMERCIAL_MORTGAGE"
}

def get_issuer(file_name):
    normalized = re.sub(
        r"[^A-Z0-9]+",
        "_",
        str(file_name).upper()
    )

    for issuer, prefix in ISSUER_PREFIXES.items():
        if prefix in normalized:
            return issuer

    return "UNKNOWN"


def parse_expected_issuers(value):
    return {
        x.strip().upper()
        for x in str(value).split(";")
        if x.strip()
    }


all_files = {
    c["file_name"] for c in ordered_chunks
}

complete_files = set(
    coverage_df.loc[
        coverage_df["fully_indexed"],
        "file_name"
    ]
)

indexed_issuers = {
    get_issuer(f) for f in complete_files
}

# These questions specifically concern one Toyota filing
TARGET_FILE_QUESTIONS = {"q17", "q18"}

coverage_rows = []

for _, row in questions_df.iterrows():
    qid = row["id"]
    expected = parse_expected_issuers(
        row["expected_issuers"]
    )

    covered_issuers = expected & indexed_issuers
    missing_issuers = expected - indexed_issuers

    missing_filings = sum(
        get_issuer(f) in expected
        for f in all_files - complete_files
    )

    if not expected:
        status = "negative_unverified"

    elif qid in TARGET_FILE_QUESTIONS:
        status = (
            "ready_target_filing"
            if TOYOTA_FILE in complete_files
            else "not_covered"
        )

    elif not covered_issuers:
        status = "not_covered"

    elif missing_issuers or missing_filings:
        status = "partial_coverage"

    else:
        status = "covered_expected_issuers"

    coverage_rows.append({
        "id": qid,
        "question": row["question"],
        "expected_issuers": "; ".join(sorted(expected)),
        "covered_issuers": "; ".join(
            sorted(covered_issuers)
        ),
        "missing_issuers": "; ".join(
            sorted(missing_issuers)
        ),
        "missing_expected_filings": missing_filings,
        "coverage_status": status
    })

eval_coverage_df = pd.DataFrame(coverage_rows)

display(eval_coverage_df[
    [
        "id", "question",
        "covered_issuers",
        "coverage_status"
    ]
])

print("\nCoverage summary:")
print(eval_coverage_df["coverage_status"].value_counts())

print("\nTotal questions:", len(eval_coverage_df))
print("Fully indexed filings:", len(complete_files))
print("Total filings:", len(all_files))


,id,question,covered_issuers,coverage_status
0,q01,Which deals hold some contracts in paper (phys...,TOYOTA,partial_coverage
1,q02,Which deals keep electronic contracts with a t...,AFS SENSUB; TOYOTA,partial_coverage
2,q03,Which deals are backed by motorcycle loans?,,not_covered
3,q04,Which deals securitize vehicle leases instead ...,,not_covered
4,q05,Which deals are backed by credit card receivab...,AMERICAN EXPRESS,partial_coverage
5,q06,Which deals are backed by cell phone device pa...,,not_covered
6,q07,Which deals are backed by commercial mortgage ...,,not_covered
7,q08,Which deals finance used cars sold at CarMax s...,,not_covered
8,q09,Which deals are sponsored by GM Financial (Ame...,AFS SENSUB,covered_expected_issuers
9,q10,Which deals are sponsored by Toyota Motor Cred...,TOYOTA,partial_coverage



Coverage summary:
coverage_status
not_covered                 11
partial_coverage             5
ready_target_filing          2
covered_expected_issuers     1
negative_unverified          1
Name: count, dtype: int64

Total questions: 20
Fully indexed filings: 12
Total filings: 122


In [139]:
# The legacy retrieval snapshots are fixed inputs for before/after replay.
TESTABLE_STATUSES = {
    "ready_target_filing",
    "partial_coverage",
    "covered_expected_issuers",
}

EVAL_RAW = {}
for _, row in eval_coverage_df.iterrows():
    qid = row["id"]
    if row["coverage_status"] not in TESTABLE_STATUSES:
        print(f"{qid}: skipped ({row['coverage_status']})")
        continue
    cache_path = EVAL_DIR / f"{qid}_index_{index.ntotal}_retrieval.json"
    if cache_path.exists():
        EVAL_RAW[qid] = pd.DataFrame(json.loads(cache_path.read_text(encoding="utf-8")))
        print(f"{qid}: loaded saved retrieval")
    else:
        print(f"{qid}: eligible, awaiting evaluation evidence")

print("Evaluation queries available:", len(EVAL_RAW))


q01: loaded saved retrieval
q02: loaded saved retrieval
q03: skipped (not_covered)
q04: skipped (not_covered)
q05: loaded saved retrieval
q06: skipped (not_covered)
q07: skipped (not_covered)
q08: skipped (not_covered)
q09: loaded saved retrieval
q10: loaded saved retrieval
q11: skipped (not_covered)
q12: skipped (not_covered)
q13: skipped (not_covered)
q14: loaded saved retrieval
q15: skipped (not_covered)
q16: skipped (not_covered)
q17: loaded saved retrieval
q18: loaded saved retrieval
q19: skipped (negative_unverified)
q20: skipped (not_covered)

Evaluation queries available: 8
New API requests this run: 0


In [140]:
# Manually identified evidence from our Toyota review
GOLD_TOYOTA_CHUNKS = {
    "q17": {212},
    "q18": {63, 228, 229, 239}
}

metric_rows = []
review_rows = []

for _, question_row in eval_coverage_df.iterrows():

    qid = question_row["id"]
    expected = parse_expected_issuers(
        question_row["expected_issuers"]
    )
    covered_expected = expected & indexed_issuers

    metric = {
        "id": qid,
        "question": question_row["question"],
        "coverage_status":
            question_row["coverage_status"],
        "retrieval_tested": False,
        "issuer_hit_at_5": np.nan,
        "issuer_recall_at_5": np.nan,
        "gold_chunk_hit_at_8": np.nan,
        "gold_chunk_recall_at_8": np.nan
    }

    if qid not in EVAL_RAW:
        metric_rows.append(metric)
        continue

    hits = EVAL_RAW[qid].copy()

    if hits.empty:
        metric["retrieval_tested"] = True
        if qid in TARGET_FILE_QUESTIONS:
            metric.update(gold_chunk_hit_at_8=0, gold_chunk_recall_at_8=0.0)
        elif covered_expected:
            metric.update(issuer_hit_at_5=0, issuer_recall_at_5=0.0)
        metric_rows.append(metric)
        continue

    metric["retrieval_tested"] = True

    hits["issuer"] = hits["file_name"].map(
        get_issuer
    )

    if qid in TARGET_FILE_QUESTIONS:
        # Evaluate relevant passages within one Toyota filing
        selected = hits.head(8).copy()

        selected["chunk_number"] = (
            selected["chunk_id"]
            .str.extract(r"_chunk_(\d+)$")
            .astype(int)
        )

        gold = GOLD_TOYOTA_CHUNKS[qid]

        retrieved_numbers = set(
            selected.loc[selected["file_name"] == TOYOTA_FILE, "chunk_number"]
        )

        matched_gold = gold & retrieved_numbers

        metric["gold_chunk_hit_at_8"] = int(
            len(matched_gold) > 0
        )

        metric["gold_chunk_recall_at_8"] = (
            len(matched_gold) / len(gold)
        )

    else:
        # Show each issuer once, ranked by its
        # highest-scoring retrieved chunk
        selected = (
            hits.drop_duplicates(subset="issuer")
            .head(5)
            .copy()
        )

        found = set(selected["issuer"])

        if covered_expected:
            metric["issuer_hit_at_5"] = int(
                bool(found & covered_expected)
            )

            metric["issuer_recall_at_5"] = (
                len(found & covered_expected)
                / len(covered_expected)
            )

    # Create rows for manual evidence review
    for rank, (_, hit) in enumerate(
        selected.iterrows(), 1
    ):

        review_rows.append({
            "question_id": qid,
            "question": question_row["question"],
            "rank": rank,
            "issuer": hit["issuer"],
            "file_name": hit["file_name"],
            "chunk_id": hit["chunk_id"],
            "doc_url": hit["doc_url"],
            "passage": hit["text"],
            "relevance_label": "",
            "review_notes": ""
        })

    metric_rows.append(metric)


m2_metrics_df = pd.DataFrame(metric_rows)
review_columns = [
    "question_id", "question", "rank", "issuer", "file_name", "chunk_id",
    "doc_url", "passage", "relevance_label", "review_notes"
]
m2_review_df = pd.DataFrame(review_rows, columns=review_columns)

metrics_path = EVAL_DIR / "retrieval_metrics.csv"

retrieval_review_path = (
    EVAL_DIR /
    f"retrieval_manual_review_index_{index.ntotal}.csv"
)

if not metrics_path.exists():
    m2_metrics_df.to_csv(metrics_path, index=False)

# Preserve any previous manual labels
if not retrieval_review_path.exists():
    m2_review_df.to_csv(
        retrieval_review_path,
        index=False
    )

print("Retrieval evaluation summary")
print("-" * 45)
print("Total questions:", len(m2_metrics_df))
print(
    "Questions tested:",
    int(m2_metrics_df["retrieval_tested"].sum())
)

display(
    m2_metrics_df[
        [
            "id",
            "coverage_status",
            "retrieval_tested",
            "issuer_hit_at_5",
            "issuer_recall_at_5",
            "gold_chunk_hit_at_8",
            "gold_chunk_recall_at_8"
        ]
    ]
)

print("\nSaved metrics:", metrics_path)
print("Manual review file:", retrieval_review_path)


Retrieval evaluation summary
---------------------------------------------
Total questions: 20
Questions tested: 8


,id,coverage_status,retrieval_tested,issuer_hit_at_5,issuer_recall_at_5,gold_chunk_hit_at_8,gold_chunk_recall_at_8
0,q01,partial_coverage,True,1.0,1.0,NaN,NaN
1,q02,partial_coverage,True,1.0,1.0,NaN,NaN
2,q03,not_covered,False,NaN,NaN,NaN,NaN
3,q04,not_covered,False,NaN,NaN,NaN,NaN
4,q05,partial_coverage,True,1.0,1.0,NaN,NaN
5,q06,not_covered,False,NaN,NaN,NaN,NaN
6,q07,not_covered,False,NaN,NaN,NaN,NaN
7,q08,not_covered,False,NaN,NaN,NaN,NaN
8,q09,covered_expected_issuers,True,1.0,1.0,NaN,NaN
9,q10,partial_coverage,True,1.0,1.0,NaN,NaN



Saved metrics: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/m2_evaluation/retrieval_metrics.csv
Manual review file: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/m2_evaluation/retrieval_manual_review_index_2808.csv


In [142]:
review_columns = [
    "question_id", "question", "rank", "issuer",
    "file_name", "chunk_id", "doc_url", "passage",
    "relevance_label", "review_notes"
]

# Read existing review data without losing manual labels
if retrieval_review_path.exists():
    try:
        reviewed = pd.read_csv(retrieval_review_path)
    except pd.errors.EmptyDataError:
        reviewed = pd.DataFrame(columns=review_columns)
else:
    reviewed = pd.DataFrame(columns=review_columns)

# Restore review rows only if the existing file has no data
if reviewed.empty:
    if not m2_review_df.empty:
        reviewed = m2_review_df.copy()

    # Save a valid CSV with headers
    reviewed.to_csv(retrieval_review_path, index=False)

print("Review file:", retrieval_review_path)
print("Passages available for review:", len(reviewed))

manual_scores = []

if not reviewed.empty:

    reviewed["relevance_label"] = pd.to_numeric(
        reviewed["relevance_label"],
        errors="coerce"
    )

    for qid, group in reviewed.groupby("question_id"):

        top5 = group.sort_values("rank").head(5)

        # Require five manually labeled passages
        if len(top5) < 5:
            continue

        if top5["relevance_label"].isna().any():
            continue

        precision = (
            top5["relevance_label"] == 2
        ).mean()

        manual_scores.append({
            "question_id": qid,
            "reviewed_passages": 5,
            "precision_at_5": round(float(precision), 3)
        })

manual_metrics_df = pd.DataFrame(
    manual_scores,
    columns=[
        "question_id",
        "reviewed_passages",
        "precision_at_5"
    ]
)

if not manual_metrics_df.empty:
    display(manual_metrics_df)

    print(
        "Mean reviewed Precision@5:",
        round(manual_metrics_df["precision_at_5"].mean(), 3)
    )
else:
    print(
        "No fully labeled questions yet. "
        "Fill in relevance_label in the review CSV."
    )

manual_metrics_path = EVAL_DIR / "manual_precision_results.csv"
if not manual_metrics_path.exists():
    manual_metrics_df.to_csv(manual_metrics_path, index=False)

print("Manual evaluation cell completed successfully.")

Review file: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/m2_evaluation/retrieval_manual_review_index_2808.csv
Passages available for review: 31
No fully labeled questions yet. Fill in relevance_label in the review CSV.
Manual evaluation cell completed successfully.


In [143]:
# The historical answer review CSV remains available on disk.
# The versioned benchmark below uses exact question, prompt, checkpoint and evidence identity.

Answers available: 2
New LLM calls: 0
Review file: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/m2_evaluation/llm_answer_manual_review.csv


,question_id,answer_source,citations_valid,has_citations
0,q17,previous_Toyota_pilot,True,True
1,q18,previous_Toyota_pilot,True,True


In [144]:
weaknesses = []

not_covered = int(
    (
        eval_coverage_df["coverage_status"]
        == "not_covered"
    ).sum()
)

partial = int(
    (
        eval_coverage_df["coverage_status"]
        == "partial_coverage"
    ).sum()
)

weaknesses.append({
    "area": "Document coverage",
    "finding": (
        f"{len(complete_files)} of {len(all_files)} "
        "filings fully indexed; "
        f"{not_covered} evaluation questions "
        "lack their expected issuers"
    ),
    "improvement": (
        "Expand document embeddings to missing issuers "
        "and rerun the evaluation."
    )
})

if partial > 0:
    weaknesses.append({
        "area": "Partial evaluation coverage",
        "finding": (
            f"{partial} questions have some relevant "
            "indexed filings but incomplete coverage."
        ),
        "improvement": (
            "Report partial-scope results separately "
            "from full-corpus performance."
        )
    })

weaknesses.append({
    "area": "True-sale interpretation",
    "finding": (
        "The initial Toyota answer began with 'No' "
        "when the evidence supported a more qualified "
        "conclusion."
    ),
    "improvement": (
        "Distinguish 'not established' from 'false' "
        "and review contractual legal conclusions."
    )
})

weaknesses.append({
    "area": "Retrieval relevance",
    "finding": (
        "An earlier cross-deal custody search returned "
        "an American Express bankruptcy passage that "
        "did not establish electronic custody."
    ),
    "improvement": (
        "Improve evidence relevance ranking and "
        "distinguish electronic custody, title "
        "recording, and bankruptcy-related control."
    )
})

weaknesses.append({
    "area": "Citation verification",
    "finding": (
        "Citation-ID checks confirm references exist "
        "but cannot prove that they support each claim."
    ),
    "improvement": (
        "Manually review source support and track "
        "unsupported or misleading citations."
    )
})

if not manual_metrics_df.empty:
    low_precision = manual_metrics_df[
        manual_metrics_df["precision_at_5"] < 0.6
    ]

    for _, row in low_precision.iterrows():
        weaknesses.append({
            "area": "Low retrieval precision",
            "finding": (
                f"{row['question_id']} had reviewed "
                f"Precision@5 = {row['precision_at_5']}."
            ),
            "improvement": (
                "Review the missed evidence and compare "
                "semantic, keyword, and hybrid rankings."
            )
        })

weakness_df = pd.DataFrame(weaknesses)

weakness_path = (
    EVAL_DIR / "milestone2_weakness_analysis.csv"
)

if not weakness_path.exists():
    weakness_df.to_csv(weakness_path, index=False)

display(weakness_df)

print("Saved weakness analysis:", weakness_path)


,area,finding,improvement
0,Document coverage,12 of 122 filings fully indexed; 11 evaluation...,Expand document embeddings to missing issuers ...
1,Partial evaluation coverage,5 questions have some relevant indexed filings...,Report partial-scope results separately from f...
2,True-sale interpretation,The initial Toyota answer began with 'No' when...,Distinguish 'not established' from 'false' and...
3,Retrieval relevance,An earlier cross-deal custody search returned ...,Improve evidence relevance ranking and disting...
4,Citation verification,Citation-ID checks confirm references exist bu...,Manually review source support and track unsup...


Saved weakness analysis: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/m2_evaluation/milestone2_weakness_analysis.csv


### Milestone 3

In [145]:
# Gradio is installed with the other notebook dependencies above.

INFO: pip is looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
  Using cached httpx2-2.15.1-py3-none-any.whl.metadata (9.6 kB)
INFO: pip is still looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 37.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 32.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [146]:
import gradio as gr
import json
from threading import Lock
from urllib.parse import urlsplit
import re

# Display only fully indexed filings
UI_FILE_CHOICES = [
    ("All fully indexed filings", "__all__")
]

for file_name in sorted(complete_files):
    UI_FILE_CHOICES.append(
        (
            file_name.replace("_", " "),
            file_name
        )
    )

# Safety limit for new queries in this notebook session
UI_MAX_NEW_QUERIES = 3
ui_new_query_count = 0
ui_budget_lock = Lock()


def format_rag_result(result):

    if result is None:
        return "No answer available.", ""

    answer = result.get("answer", "")

    sources = result.get("sources", [])

    source_lines = []

    for source in sources:

        reference = source.get("reference", "")
        chunk_id = source.get("chunk_id", "")
        file_name = source.get("file_name", "")
        url = source.get("doc_url", "")

        # Use original indexed text for short previews
        chunk = indexed_lookup.get(chunk_id, {})
        preview = re.sub(
            r"\s+", " ",
            chunk.get("text", "")
        )[:350]

        parsed_url = urlsplit(url or "")
        label = file_name
        if parsed_url.scheme == "https" and parsed_url.netloc == "www.sec.gov":
            label = f"[{file_name}]({url})"

        source_text = f"**[{reference}]** {label}\n\nChunk: `{chunk_id}`"
        if preview:
            source_text += f"\n\n> {preview}"
            if len(chunk["text"]) > 350:
                source_text += "…"
        source_lines.append(source_text)

    if not source_lines:
        source_text = "No source references available."
    else:
        source_text = "\n\n---\n\n".join(source_lines)

    unknown = result.get("unknown_citations", [])

    if unknown:
        answer += (
            "\n\n**Warning:** Unrecognized citation IDs: "
            + ", ".join(unknown)
        )

    return answer, source_text


def ask_ui_question(question, selected_file, allow_api):

    global ui_new_query_count

    question = (question or "").strip()
    if not question:
        return "Please enter a question.", ""

    file_name = (
        None if selected_file == "__all__"
        else selected_file
    )

    if file_name is not None:
        if file_name not in complete_files:
            return (
                "This filing is not fully indexed yet.",
                ""
            )

    cache_path = question_cache_path(question, file_name, TOP_K)

    is_cached = cache_path.exists()

    if not is_cached and not allow_api:
        return (
            "This question has not been answered before. "
            "Enable **Allow API calls** to generate an "
            "answer, or open a saved example.",
            ""
        )

    if not is_cached:
        if client is None:
            return "Configure API credentials to ask a new question.", ""
        with ui_budget_lock:
            if ui_new_query_count >= UI_MAX_NEW_QUERIES:
                return (
                    "The notebook's new-query API limit has been reached. "
                    "Cached answers are still available.",
                    ""
                )
            ui_new_query_count += 1

    try:
        result = answer_new_question(
            question=question,
            file_name=file_name,
            top_k=TOP_K,
            run_api=bool(allow_api)
        )
        return format_rag_result(result)
    except Exception as error:
        if getattr(error, "status_code", None) == 429:
            return "The API rate or quota limit was reached.", ""
        return f"Unable to generate the answer ({type(error).__name__}).", ""


def load_saved_example(example_id):

    if not example_id:
        return "Choose a saved example.", ""

    # Results from the existing Task 2 notebook
    result = TASK2_RESULTS.get(example_id)

    if result is None:
        saved_path = (
            TASK2_DIR / f"{example_id}_answer.json"
        )

        if saved_path.exists():
            try:
                result = json.loads(saved_path.read_text(encoding="utf-8"))
            except (OSError, ValueError):
                return "Unable to read the saved answer.", ""

    if result is None:
        return "No saved answer found for this example.", ""

    return format_rag_result(result)


print("Gradio interface functions ready.")
print("Available filings:", len(complete_files))
print("New-query API limit:", UI_MAX_NEW_QUERIES)


Gradio interface functions ready.
Available filings: 12
New-query API limit: 3


In [147]:
with gr.Blocks(title="PIMCO ABS Surveillance Assistant", analytics_enabled=False) as demo:

    gr.Markdown(
        "# PIMCO ABS Surveillance Assistant\n"
        "Ask questions about SEC asset-backed securities "
        "prospectuses and view supporting sources."
    )

    gr.Markdown(
        f"**Current coverage:** {len(complete_files)} "
        f"fully indexed filings out of {len(all_files)}. "
        "Results do not represent the entire collection."
    )

    with gr.Tab("Ask a Question"):

        question_input = gr.Textbox(
            label="Your question",
            placeholder=(
                "Example: How long can collections "
                "stay in a servicer account?"
            ),
            lines=3
        )

        file_input = gr.Dropdown(
            choices=UI_FILE_CHOICES,
            value="__all__",
            label="Prospectus selection"
        )

        allow_api = gr.Checkbox(
            value=False,
            label=(
                "Allow API calls for new questions "
                "(uses PIMCO API credits)"
            )
        )

        ask_button = gr.Button(
            "Find Answer",
            variant="primary"
        )

        answer_output = gr.Markdown(
            label="Generated Answer"
        )

        sources_output = gr.Markdown(
            label="Supporting Sources"
        )

        ask_button.click(
            fn=ask_ui_question,
            inputs=[
                question_input,
                file_input,
                allow_api
            ],
            outputs=[
                answer_output,
                sources_output
            ],
            api_visibility="private"
        )

    with gr.Tab("Saved Examples"):

        example_choices = [
            (case["question"], case["id"])
            for case in TEST_CASES
        ]

        example_input = gr.Dropdown(
            choices=example_choices,
            value=None,
            label="Select a tested question"
        )

        example_button = gr.Button(
            "Load Saved Answer"
        )

        example_answer = gr.Markdown(
            label="Answer"
        )

        example_sources = gr.Markdown(
            label="Sources"
        )

        example_button.click(
            fn=load_saved_example,
            inputs=[example_input],
            outputs=[
                example_answer,
                example_sources
            ],
            api_visibility="private"
        )

    gr.Markdown(
        "Answers are based on retrieved passages from "
        "the currently indexed filings. Verify important "
        "financial and legal conclusions against the "
        "original SEC documents."
    )

# Deprecated fallback. Set True and rerun this cell if needed.
RUN_GRADIO_FALLBACK = False
if RUN_GRADIO_FALLBACK:
    demo.launch(
        server_name="127.0.0.1",
        share=False,
        inbrowser=False,
        prevent_thread_lock=True,
        show_error=False
    )


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Manual review CSV ready: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/m2_evaluation/retrieval_manual_review_index_2808.csv


### Versioned benchmark and answer review

Run this section offline to validate the saved benchmark, replay the new true-sale evidence selection, and generate review tables. See the README for submitting human labels. The index and the historical benchmark are used as local inputs.

In [151]:
# Support kernels that ran the setup before this benchmark was added.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import prospectus_evaluation as evaluation

if "provenance_collisions" not in globals():
    from prospectus_provenance import validate_manifest

    _, provenance_collisions = validate_manifest(manifest, raw_data)

benchmark_rows, benchmark_candidates, q18_ranks = evaluation.benchmark_replay(
    eval_coverage_df.to_dict("records"),
    {qid: hits.to_dict("records") for qid, hits in EVAL_RAW.items()},
    indexed_chunks, global_vectorizer, global_tfidf, get_issuer, indexed_issuers,
    TOYOTA_FILE, GOLD_TOYOTA_CHUNKS,
)
benchmark_raw = {qid: hits.to_dict("records") for qid, hits in EVAL_RAW.items()}
benchmark_root = processed_data / "evaluation_runs"
benchmark_index = {
    "vectors": index.ntotal,
    "faiss_sha256": evaluation.file_digest(index_path),
    "chunk_archive_sha256": source_hash,
    "chunk_order_sha256": order_hash,
    "embedding_model": EMBEDDING_MODEL,
}
benchmark_review_path = (
    processed_data / "m2_evaluation"
    / f"retrieval_manual_review_index_{index.ntotal}.csv"
)
baseline_review = evaluation.passage_review(
    evaluation.review_rows(benchmark_review_path), benchmark_rows,
    benchmark_raw, indexed_chunks, get_issuer,
)
# Fixed evidence for a scoped abstention test, chosen by offline TF-IDF.
q19_question = questions_df.set_index("id").loc["q19", "question"]
full_positions = np.array([
    i for i, chunk in enumerate(indexed_chunks) if chunk["file_name"] in complete_files
])
q19_order = evaluation.keyword_order(
    global_vectorizer, global_tfidf[full_positions], q19_question,
)
q19_evidence = []
q19_seen = set()
for position in q19_order:
    chunk = indexed_chunks[full_positions[position]]
    if chunk["file_name"] not in q19_seen:
        q19_evidence.append(chunk)
        q19_seen.add(chunk["file_name"])
    if len(q19_evidence) == TOP_K:
        break
benchmark_plans = evaluation.answer_plan(
    benchmark_rows, benchmark_candidates, q19_evidence, benchmark_index,
    LLM_MODEL, PROMPT_VERSION, SYSTEM_PROMPT,
)
pilot_semantic = pd.read_csv(processed_data / "toyota_semantic_retrieval_results.csv")
pilot_semantic = pilot_semantic.loc[pilot_semantic["question"] == TRUE_SALE_QUESTION]
q18_semantic_pilot = {
    "question": TRUE_SALE_QUESTION,
    "exact_benchmark_semantic_ranks_available": False,
    "ranked_chunks": pilot_semantic[["chunk_id", "rank", "similarity_score"]].to_dict("records"),
}


In [152]:
def generate_benchmark_answer(plan):
    return generate_answer_from_passages(
        plan["question"], pd.DataFrame(plan["evidence"]), scope=plan["scope"]
    )


def run_benchmark_answers(*, approved=False, question_ids=(), max_calls=0):
    if (question_ids or max_calls) and approved is not True:
        raise PermissionError("Live answer evaluation requires explicit approval")
    return evaluation.evaluate_answers(
        benchmark_plans, benchmark_rows, benchmark_root / "answers",
        generate_benchmark_answer, question_ids=question_ids, max_calls=max_calls,
    )


def export_current_evaluation(answer_rows):
    historical = []
    for case in TEST_CASES:
        case_id = case["id"]
        if case_id not in TASK2_RESULTS:
            continue
        result = TASK2_RESULTS[case_id]
        stored_chunks = [
            indexed_lookup[source["chunk_id"]] for source in result["sources"]
        ]
        evidence = evaluation.evidence_rows(stored_chunks)
        historical.append({
            "case_id": case_id, "question": case["question"],
            "exact_pilot_question_match": case["question"] == result["question"],
            "answer": result["answer"], "stored_prompt_version": result.get("prompt_version"),
            "current_prepared_evidence_matches": list(TEST_EVIDENCE[case_id]["chunk_id"]) ==
            [source["chunk_id"] for source in result["sources"]],
            "citation_checks": json.dumps(evaluation.citation_checks(result, evidence)),
            "evidence_json": json.dumps(evidence, ensure_ascii=False),
            "pilot_review_notes": manual_reviews.get(case_id, {}).get("review_notes", ""),
        })
    weaknesses = [
        dict(component="coverage", failure="The UI searches only fully indexed filings",
             evidence=f"{len(complete_files)}/{len(all_files)} complete filings", likely_cause="Partial embedding checkpoint",
             recommendation="Budget and index missing filings", implemented="No", after="Scope shown in UI"),
        dict(component="custody pilot", failure="American Express insolvency passage surfaced for custody",
             evidence="Historical cross-deal rank 2 was labeled 0", likely_cause="Generic language about control",
             recommendation="Label benchmark passages and test custody-specific relevance", implemented="No", after="Pending"),
        dict(component="q18", failure="Cached top eight missed the identified transfer and true-sale passages",
             evidence=json.dumps(q18_ranks), likely_cause="Date/issuer wording and generic terms diluted lexical matching; exact semantic ranks unavailable",
             recommendation="Reserve lexical evidence for filing-specific true-sale questions", implemented="Yes",
             after="Compare before and after gold recall in retrieval_metrics.csv"),
        dict(component="historical true-sale answer", failure="Opening No is stronger than the legal evidence",
             evidence="Pilot answer distinguishes intended transfers, a pledge, and possible recharacterization",
             likely_cause="Binary framing of an uncertain legal question",
             recommendation="Review the exact q18 answer for qualified treatment", implemented="Review framework", after="Awaiting answer"),
        dict(component="citations", failure="Reference validity cannot establish claim support",
             evidence="Structural source checks and human support scores measure different things",
             likely_cause="Citation checks examine IDs and metadata",
             recommendation="Review each factual claim against the cited text", implemented="Review framework", after="Awaiting human review"),
        dict(component="benchmark labels", failure="Passage and answer quality are unmeasured",
             evidence=f"{len(baseline_review)} passage rows; see summary for current label counts",
             likely_cause="Manual review pending",
             recommendation="Submit passage and answer labels using review templates",
             implemented="Versioned templates and scoring", after="See summary.json"),
    ]
    return evaluation.export_benchmark(
        benchmark_root, benchmark_index, benchmark_rows, baseline_review,
        benchmark_candidates, q18_ranks, benchmark_plans, answer_rows, historical,
        provenance_collisions, q18_semantic_pilot, weaknesses,
    )


benchmark_run, benchmark_summary, passage_review_table, answer_review_table = (
    export_current_evaluation(run_benchmark_answers())
)
print("Evaluation export:", benchmark_run)
display(pd.DataFrame(benchmark_rows))
display(pd.DataFrame(passage_review_table)[[
    "question_id", "question", "rank", "source_ref", "file_name", "doc_url",
    "passage", "relevance_label", "review_notes",
]].head(5))
print(json.dumps(benchmark_summary, indent=2))


Evaluation export: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/evaluation_runs/benchmark_854b94df6f8980b9


,question_id,question,coverage_status,evaluated,metric,ground_truth_count,before_hit,before_recall,after_hit,after_recall
0,q01,Which deals hold some contracts in paper (phys...,partial_coverage,True,covered_issuer@5,1,1.0,1.0,1.0,1.0
1,q02,Which deals keep electronic contracts with a t...,partial_coverage,True,covered_issuer@5,2,1.0,1.0,1.0,1.0
2,q03,Which deals are backed by motorcycle loans?,not_covered,False,covered_issuer@5,0,NaN,NaN,NaN,NaN
3,q04,Which deals securitize vehicle leases instead ...,not_covered,False,covered_issuer@5,0,NaN,NaN,NaN,NaN
4,q05,Which deals are backed by credit card receivab...,partial_coverage,True,covered_issuer@5,1,1.0,1.0,1.0,1.0
5,q06,Which deals are backed by cell phone device pa...,not_covered,False,covered_issuer@5,0,NaN,NaN,NaN,NaN
6,q07,Which deals are backed by commercial mortgage ...,not_covered,False,covered_issuer@5,0,NaN,NaN,NaN,NaN
7,q08,Which deals finance used cars sold at CarMax s...,not_covered,False,covered_issuer@5,0,NaN,NaN,NaN,NaN
8,q09,Which deals are sponsored by GM Financial (Ame...,covered_expected_issuers,True,covered_issuer@5,1,1.0,1.0,1.0,1.0
9,q10,Which deals are sponsored by Toyota Motor Cred...,partial_coverage,True,covered_issuer@5,1,1.0,1.0,1.0,1.0


,question_id,question,rank,source_ref,file_name,doc_url,passage,relevance_label,review_notes
0,q01,Which deals hold some contracts in paper (phys...,1,S1,AFS_SENSUB_CORP__2026-04-03_424H.htm,https://www.sec.gov/Archives/edgar/data/134718...,are evidenced either by retail installment sal...,,
1,q01,Which deals hold some contracts in paper (phys...,2,S2,AMERICAN_EXPRESS_CREDIT_ACCOUNT_MASTER_TRUST_2...,https://www.sec.gov/Archives/edgar/data/100350...,of interest and principal in the manner descri...,,
2,q01,Which deals hold some contracts in paper (phys...,3,S3,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08...,https://www.sec.gov/Archives/edgar/data/113113...,contract if (a) there is a “single authoritati...,,
3,q02,Which deals keep electronic contracts with a t...,1,S1,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08...,https://www.sec.gov/Archives/edgar/data/113113...,contract if (a) there is a “single authoritati...,,
4,q02,Which deals keep electronic contracts with a t...,2,S2,AFS_SENSUB_CORP__2025-05-01_424H.htm,https://www.sec.gov/Archives/edgar/data/134718...,the required level of authority. Dealers are c...,,


{
  "evaluated": 8,
  "unevaluated": 12,
  "precision_at_5": {},
  "mean_precision_at_5": null,
  "retrieval": {
    "covered_issuer@5": {
      "evaluated": 6,
      "before_mean_hit": 1.0,
      "before_mean_recall": 1.0,
      "after_mean_hit": 1.0,
      "after_mean_recall": 1.0
    },
    "gold_passage@8": {
      "evaluated": 2,
      "before_mean_hit": 0.5,
      "before_mean_recall": 0.5,
      "after_mean_hit": 1.0,
      "after_mean_recall": 1.0
    }
  },
  "q18_improved_precision_at_5": null,
  "answers_awaiting_generation": 9,
  "answers_ready_for_review": 0,
  "answers_fully_reviewed": 0
}


In [153]:
def run_pimco_smoke_test(*, approved=False):
    """Run at most two embeddings and two completions against the configured endpoint."""
    if approved is not True:
        raise PermissionError("Live smoke test requires explicit approval")
    if not os.getenv("OPENAI_BASE_URL"):
        raise RuntimeError("Set the team-approved PIMCO endpoint before running the smoke test")
    if TOYOTA_FILE not in complete_files:
        raise RuntimeError("The targeted Toyota filing must be fully indexed")
    api = require_api_client()
    if api.max_retries != 0:
        raise RuntimeError("Use a client with retries disabled to enforce the request cap")
    scopes = [
        ("In the April 2026 Toyota deal, how long can collections stay in a servicer account?", TOYOTA_FILE),
        ("Which indexed deals describe third-party control of electronic loan contracts?", None),
    ]
    return evaluation.smoke_test(
        benchmark_root / "live_smoke", benchmark_index, scopes,
        lambda question, file: hybrid_search(
            question, file_name=file, top_k=TOP_K, unique_filings=file is None
        ).to_dict("records"),
        lambda question, file, passages: generate_answer_from_passages(
            question, pd.DataFrame(passages),
            scope=f"Only the selected filing {file}" if file else "Only fully indexed filings",
        ),
        approved=approved,
    )


### Milestone 3 Task 2 — Scoped benchmark

Run the cells below after the versioned benchmark. The default run validates the indexed-scope benchmark offline. Human labels go into the separate input files described in the README.

In [ ]:
import prospectus_task2 as task2

task2_version, task2_cases = task2.load_cases(
    PROJECT_ROOT / "eval" / "task2_questions_v1.json",
    questions_df.to_dict("records"), TOYOTA_FILE, GOLD_TOYOTA_CHUNKS,
)
task2_directory = benchmark_root / task2_version / benchmark_index["faiss_sha256"][:16]


def run_task2_offline():
    semantic = task2.semantic_snapshots(
        task2_cases, benchmark_index, task2_directory / "semantic",
        indexed_lookup, complete_files,
    )
    review, proposals, comparisons, selected = task2.replay(
        task2_cases, indexed_chunks, complete_files, global_vectorizer, global_tfidf,
        get_issuer, benchmark_raw, benchmark_candidates, semantic,
    )
    # Carry forward manually rated passages when the question and exact chunk match.
    historical_labels = {
        (row["question_id"], row["question"], row["chunk_id"]): row
        for row in passage_review_table if row["relevance_label"] in ("0", "1", "2")
    }
    for row in review:
        prior = historical_labels.get((row["question_id"], row["question"], row["chunk_id"]))
        if prior is not None:
            row.update(relevance_label=prior["relevance_label"], review_notes=prior["review_notes"])
    metrics = [dict(question_id=case["id"], question=case["question"]) for case in task2_cases]
    plans = evaluation.answer_plan(
        metrics, selected, q19_evidence, benchmark_index, LLM_MODEL, PROMPT_VERSION, SYSTEM_PROMPT,
    )
    return review, proposals, comparisons, metrics, plans


task2_review, task2_proposals, task2_comparisons, task2_metrics, task2_plans = run_task2_offline()


In [ ]:
def run_task2_answers(*, approved=False, question_ids=(), max_calls=0):
    if (question_ids or max_calls) and approved is not True:
        raise PermissionError("Live Task 2 answers require explicit approval")
    answers = evaluation.evaluate_answers(
        task2_plans, task2_metrics, benchmark_root / "answers",
        generate_benchmark_answer, question_ids=question_ids, max_calls=max_calls,
    )
    return task2.export(
        task2_directory, benchmark_index, task2_version, task2_cases, task2_review,
        task2_proposals, task2_comparisons, answers, complete_files,
    )


task2_export, task2_summary = run_task2_answers()
print("Task 2 export:", task2_export)
print(json.dumps(task2_summary, indent=2))
display(pd.DataFrame(task2_comparisons))


In [ ]:
def retrieve_benchmark_semantic(question, file_name):
    partial_count = sum(chunk["file_name"] not in complete_files for chunk in indexed_chunks)
    hits = semantic_search(question, file_name=file_name, top_k=8 if file_name else 8 + partial_count)
    if hits.empty:
        return []
    return hits[hits["file_name"].isin(complete_files)].head(8).to_dict("records")


def run_task2_semantic(*, approved=False, question_ids=(), max_calls=0):
    if approved is not True:
        raise PermissionError("Live semantic retrieval requires explicit approval")
    if require_api_client().max_retries != 0:
        raise RuntimeError("Use a client with retries disabled to enforce the request cap")

    return task2.run_semantic_plan(
        task2_cases, benchmark_index, task2_directory / "semantic", retrieve_benchmark_semantic,
        indexed_lookup, complete_files, approved=approved, question_ids=question_ids, max_calls=max_calls,
    )

In [ ]:
def run_heldout_semantic(manifest_path, *, approved=False, question_ids=(), max_calls=0):
    """Evaluate an approved held-out proposal set separately from development runs."""
    if approved is not True:
        raise PermissionError("Live held-out retrieval requires explicit API approval")
    manifest_path = Path(manifest_path).resolve()
    expected_directory = (
        benchmark_root / "heldout_proposals_v1" / benchmark_index["faiss_sha256"][:16]
    ).resolve()
    if manifest_path.parent != expected_directory:
        raise ValueError("Select the approved held-out manifest from Evaluation Review")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if (manifest_path.name != f"approved_{evaluation.digest(manifest)[:16]}.json"
            or manifest["index"] != benchmark_index
            or manifest["version"] != "heldout_proposals_v1"):
        raise ValueError("Held-out manifest identity or checkpoint does not match")
    if require_api_client().max_retries != 0:
        raise RuntimeError("Use a client with retries disabled to enforce the request cap")
    return task2.run_semantic_plan(
        manifest["cases"], benchmark_index,
        expected_directory / manifest_path.stem / "semantic",
        retrieve_benchmark_semantic, indexed_lookup, complete_files,
        approved=approved, question_ids=question_ids, max_calls=max_calls,
    )
